In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

In [3]:
pd.set_option("display.max_rows", None)

In [329]:
DATA_DIR = "data/processed/"
daily_data = pd.read_csv(
    DATA_DIR + "canada_integrated_daily_macro_features_final.csv",
    parse_dates=[
        "Date",
        "Bank_Availability_Date",
        "Macro_Reference_Date",
        "Macro_Availability_Date"
    ]
)

daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

print("Shape:", daily_data.shape)
print("Tickers:", daily_data["Ticker"].nunique())
print("Date Range:", daily_data["Date"].min(), "to", daily_data["Date"].max())

daily_data.head()

Shape: (63574, 68)
Tickers: 10
Date Range: 2000-01-04 00:00:00 to 2026-11-06 00:00:00


,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,...,Close_MA_Ratio_20,Close_MA_Ratio_50,Close_Momentum_5D,Close_Momentum_10D,Close_Momentum_20D,Close_RSI_14,MACD_Pct,MACD_Signal_Pct,MACD_Histogram_Pct,Policy_Rate_Change_21D
0,2000-01-04,ABX.TO,25.150000,25.65,24.950001,25.450001,17.804482,926500,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-05,ABX.TO,25.400000,26.00,25.350000,25.650000,17.944401,1452200,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-06,ABX.TO,25.799999,26.00,25.500000,25.799999,18.049345,600200,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-07,ABX.TO,25.850000,26.00,25.100000,25.250000,17.664574,1020800,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-10,ABX.TO,25.350000,25.35,25.150000,25.250000,17.664574,1281100,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [330]:
# Display all columns
for i, column in enumerate(daily_data.columns, start=1):
    print(f"{i:02d}. {column}")

01. Date
02. Ticker
03. Open
04. High
05. Low
06. Close
07. Adj Close
08. Volume
09. Dividends
10. Stock Splits
11. Bank_Availability_Date
12. Policy_Rate
13. Bond_Yield_10Y
14. USD_CAD
15. Macro_Reference_Date
16. CPI
17. Inflation_YoY
18. GDP_Real_Millions_CAD
19. GDP_MoM
20. GDP_YoY
21. Employment
22. Labour_Force
23. Participation_Rate
24. Unemployment_Rate
25. BCPI_Total
26. BCPI_NonEnergy
27. BCPI_Energy
28. BCPI_Metals
29. BCPI_Forestry
30. BCPI_Agriculture
31. BCPI_Fish
32. Macro_Availability_Date
33. DayOfWeek
34. Month
35. Quarter
36. DayOfWeek_Sin
37. DayOfWeek_Cos
38. Month_Sin
39. Month_Cos
40. Macro_Feature_Availability_Date
41. BCPI_Change_12M
42. BCPI_Energy_Change_12M
43. BCPI_Metals_Change_12M
44. Unemployment_Change_12M
45. Employment_Growth_12M
46. Yield_Spread
47. USD_CAD_Change_21D
48. Future_Return_5D
49. Close_Daily_Return
50. Close_Return_Lag_1
51. Close_Return_Lag_5
52. Close_Return_Lag_10
53. Close_Return_Lag_20
54. Close_Volatility_5D
55. Close_Volatility_10

In [9]:
missing_summary = pd.DataFrame({
    "Missing_Count": daily_data.isnull().sum(),
    "Missing_Percentage": (
        daily_data.isnull().mean() * 100
    )
})

missing_summary = missing_summary.sort_values(
    "Missing_Percentage",
    ascending=False
)

missing_summary

,Missing_Count,Missing_Percentage
USD_CAD_Change_21D,39324,61.855476
USD_CAD,38984,61.320666
Policy_Rate_Change_21D,21299,33.502690
Yield_Spread,21294,33.494825
Policy_Rate,21090,33.173939
Bond_Yield_10Y,2473,3.889955
Bank_Availability_Date,2259,3.553339
Close_MA_Ratio_50,490,0.770755
MACD_Signal_Pct,330,0.519080
MACD_Histogram_Pct,330,0.519080


In [5]:
RAW = [
    "Open", "High", "Low", "Close",
    "Adj Close", "Volume",
    "Dividends", "Stock Splits"
]

In [6]:
META = [
    "Date",
    "Ticker",
    "Bank_Availability_Date",
    "Macro_Reference_Date",
    "Macro_Availability_Date",
    "Macro_Feature_Availability_Date"
]

In [7]:
# Economic indicators
ECONOMIC = [
    "Policy_Rate",
    "Bond_Yield_10Y",
    "USD_CAD",
    "CPI",
    "Inflation_YoY",
    "GDP_Real_Millions_CAD",
    "GDP_MoM",
    "GDP_YoY",
    "Employment",
    "Labour_Force",
    "Participation_Rate",
    "Unemployment_Rate",
    "BCPI_Total",
    "BCPI_NonEnergy",
    "BCPI_Energy",
    "BCPI_Metals",
    "BCPI_Forestry",
    "BCPI_Agriculture",
    "BCPI_Fish",
    "BCPI_Change_12M",
    "BCPI_Energy_Change_12M",
    "BCPI_Metals_Change_12M",
    "Unemployment_Change_12M",
    "Employment_Growth_12M",
    "Yield_Spread",
    "Policy_Rate_Change_21D",
    "USD_CAD_Change_21D"
]

In [8]:
# Cyclical calendar features
CALENDAR = [
    "DayOfWeek_Sin",
    "DayOfWeek_Cos",
    "Month_Sin",
    "Month_Cos"
]

In [9]:
# Technical indicators
TECHNICAL = [
    "Close_Daily_Return",
    "Close_Return_Lag_1",
    "Close_Return_Lag_5",
    "Close_Return_Lag_10",
    "Close_Return_Lag_20",
    "Close_Volatility_5D",
    "Close_Volatility_10D",
    "Close_Volatility_20D",
    "Close_MA_Ratio_5",
    "Close_MA_Ratio_10",
    "Close_MA_Ratio_20",
    "Close_MA_Ratio_50",
    "Close_Momentum_5D",
    "Close_Momentum_10D",
    "Close_Momentum_20D",
    "Close_RSI_14",
    "MACD_Pct",
    "MACD_Signal_Pct",
    "MACD_Histogram_Pct"
]

In [10]:
# Categorical features to be one-hot encoded
DUMMIES = [
    "Ticker"
]

In [11]:
BANK = [
    "Policy_Rate",
    "Bond_Yield_10Y",
    "USD_CAD",
    "Yield_Spread",
    "Policy_Rate_Change_21D",
    "USD_CAD_Change_21D"
]


In [12]:
TO_PREDICT = "Future_Return_5D"

In [13]:
# Modeling Features
NUMERIC = TECHNICAL + ECONOMIC + CALENDAR
FEATURES = NUMERIC + DUMMIES

In [13]:
target_column = "Future_Return_5D"

In [19]:
missing_summary = pd.DataFrame({
    "Missing_Count": daily_data.isnull().sum(),
    "Missing_Percentage": daily_data.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(
    "Missing_Count",
    ascending=False
)

display(missing_summary)

,Missing_Count,Missing_Percentage
USD_CAD_Change_21D,39324,61.855476
USD_CAD,38984,61.320666
Policy_Rate_Change_21D,21299,33.502690
Yield_Spread,21294,33.494825
Policy_Rate,21090,33.173939
Bond_Yield_10Y,2473,3.889955
Bank_Availability_Date,2259,3.553339
Close_MA_Ratio_50,490,0.770755
MACD_Signal_Pct,330,0.519080
MACD_Histogram_Pct,330,0.519080


In [15]:
# Missing values by ticker
missing_by_ticker = (
    daily_data.groupby("Ticker")[BANK]
    .apply(lambda x: x.isnull().sum())
)

display(missing_by_ticker)

,Policy_Rate,Bond_Yield_10Y,USD_CAD,Yield_Spread,Policy_Rate_Change_21D,USD_CAD_Change_21D
Ticker,,,,,,
ABX.TO,2343,273,4285,2364,2366,4319
BMO.TO,2343,273,4285,2364,2366,4319
BN.TO,2343,273,4285,2364,2366,4319
CNQ.TO,2343,273,4285,2364,2366,4319
CNR.TO,2343,273,4285,2364,2366,4319
ENB.TO,2343,273,4285,2364,2366,4319
RY.TO,2343,273,4285,2364,2366,4319
SHOP.TO,3,16,419,18,5,453
SU.TO,2343,273,4285,2364,2366,4319


In [21]:
for col in BANK:

    available = daily_data.loc[
        daily_data[col].notna(),
        ["Date", "Ticker"]
    ]

    print(f"\n{'=' * 50}")
    print(f"Feature: {col}")
    print(f"First available: {available['Date'].min()}")
    print(f"Last available: {available['Date'].max()}")
    print(f"Available rows: {len(available)}")


Feature: Policy_Rate
First available: 2009-04-21 00:00:00
Last available: 2026-11-05 00:00:00
Available rows: 42484

Feature: Bond_Yield_10Y
First available: 2001-01-02 00:00:00
Last available: 2026-11-05 00:00:00
Available rows: 61101

Feature: USD_CAD
First available: 2017-01-03 00:00:00
Last available: 2026-11-06 00:00:00
Available rows: 24590

Feature: Yield_Spread
First available: 2009-04-21 00:00:00
Last available: 2026-11-05 00:00:00
Available rows: 42280

Feature: Policy_Rate_Change_21D
First available: 2009-05-21 00:00:00
Last available: 2026-11-05 00:00:00
Available rows: 42275

Feature: USD_CAD_Change_21D
First available: 2017-02-01 00:00:00
Last available: 2026-11-06 00:00:00
Available rows: 24250


In [20]:
# Check whether economic values are consistent across tickers on each date

economic_consistency = (
    daily_data
    .groupby("Date")[ECONOMIC]
    .nunique(dropna=True)
)

inconsistent_features = economic_consistency.columns[
    (economic_consistency > 1).any()
].tolist()

print("Economic features with inconsistent values across tickers:")
print(inconsistent_features)

if not inconsistent_features:
    print("All economic features are consistent across tickers for each date.")

Economic features with inconsistent values across tickers:
[]
All economic features are consistent across tickers for each date.


In [21]:
economic_data = (
    daily_data[["Date"] + ECONOMIC]
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .set_index("Date")
)

print("Economic data shape:", economic_data.shape)
print("Unique dates:", economic_data.index.nunique())

Economic data shape: (6744, 27)
Unique dates: 6744


In [24]:
exact_duplicates = []

for i, col1 in enumerate(ECONOMIC):
    for col2 in ECONOMIC[i + 1:]:
        if economic_data[col1].equals(economic_data[col2]):
            exact_duplicates.append((cmetadata_columnsol1, col2))

print("Exactly identical economic feature pairs:")

if exact_duplicates:
    for col1, col2 in exact_duplicates:
        print(f"{col1} == {col2}")
else:
    print("No exactly identical feature pairs found.")

Exactly identical economic feature pairs:
No exactly identical feature pairs found.


# Data Collection & Integration

## Stock Market Data Correlation

In [121]:
stock_corr = (
    daily_data
    .select_dtypes(include="number")
    .corr(method="pearson")
)

print("Stock Feature Correlation Matrix:")
display(stock_corr.round(2))

Stock Feature Correlation Matrix:


,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Policy_Rate,Bond_Yield_10Y,...,Close_MA_Ratio_50,Close_Momentum_5D,Close_Momentum_10D,Close_Momentum_20D,Close_RSI_14,MACD_Pct,MACD_Signal_Pct,MACD_Histogram_Pct,Future_Outperformance_5D,Policy_Rate_Change_21D
Open,1.00,1.00,1.00,1.00,0.98,-0.14,0.07,-0.01,0.28,-0.22,...,0.05,0.01,0.02,0.03,0.07,0.06,0.06,0.00,0.01,-0.01
High,1.00,1.00,1.00,1.00,0.98,-0.14,0.07,-0.01,0.28,-0.22,...,0.05,0.01,0.02,0.03,0.07,0.05,0.06,0.00,0.01,-0.01
Low,1.00,1.00,1.00,1.00,0.98,-0.15,0.07,-0.01,0.28,-0.22,...,0.05,0.01,0.02,0.03,0.08,0.06,0.06,0.01,0.01,-0.01
Close,1.00,1.00,1.00,1.00,0.98,-0.14,0.07,-0.01,0.28,-0.22,...,0.05,0.02,0.02,0.03,0.08,0.06,0.06,0.01,0.01,-0.01
Adj Close,0.98,0.98,0.98,0.98,1.00,-0.10,0.06,-0.01,0.35,-0.20,...,0.06,0.02,0.03,0.04,0.07,0.06,0.06,0.00,0.02,-0.03
Volume,-0.14,-0.14,-0.15,-0.14,-0.10,1.00,0.06,0.00,0.15,-0.01,...,-0.02,-0.03,-0.03,-0.02,-0.04,-0.07,-0.06,-0.05,0.02,-0.03
Dividends,0.07,0.07,0.07,0.07,0.06,0.06,1.00,0.00,0.02,-0.02,...,-0.01,-0.02,-0.01,-0.00,-0.02,0.01,0.01,-0.00,0.00,0.00
Stock Splits,-0.01,-0.01,-0.01,-0.01,-0.01,0.00,0.00,1.00,0.00,0.01,...,-0.00,-0.00,0.00,-0.00,-0.00,-0.00,-0.00,0.01,0.00,0.01
Policy_Rate,0.28,0.28,0.28,0.28,0.35,0.15,0.02,0.00,1.00,0.60,...,-0.00,-0.00,-0.00,-0.00,-0.02,0.01,0.01,-0.00,0.02,0.05
Bond_Yield_10Y,-0.22,-0.22,-0.22,-0.22,-0.20,-0.01,-0.02,0.01,0.60,1.00,...,0.01,0.00,-0.00,-0.00,-0.00,0.03,0.03,-0.01,-0.03,0.14


In [122]:
correlation_threshold = 0.90

upper_triangle = stock_corr.where(
    np.triu(
        np.ones(stock_corr.shape),
        k=1
    ).astype(bool)
)

high_correlations = (
    upper_triangle
    .stack()
    .reset_index()
)

high_correlations.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

high_correlations["Absolute_Correlation"] = (
    high_correlations["Correlation"].abs()
)

high_correlations = (
    high_correlations[
        high_correlations["Absolute_Correlation"] >= correlation_threshold
    ]
    .sort_values("Absolute_Correlation", ascending=False)
    .reset_index(drop=True)
)

print("Highly Correlated Stock Feature Pairs:")
display(high_correlations.round(4))

Highly Correlated Stock Feature Pairs:


,Feature_1,Feature_2,Correlation,Absolute_Correlation
0,Open,High,0.9999,0.9999
1,Low,Close,0.9999,0.9999
2,High,Close,0.9999,0.9999
3,Open,Low,0.9998,0.9998
4,High,Low,0.9998,0.9998
5,Open,Close,0.9998,0.9998
6,Employment,Labour_Force,0.9928,0.9928
7,GDP_Real_Millions_CAD,Employment,0.9920,0.9920
8,CPI,Labour_Force,0.9901,0.9901
9,GDP_Real_Millions_CAD,Labour_Force,0.9881,0.9881


In [123]:
all_correlations = (
    upper_triangle
    .stack()
    .reset_index()
)

all_correlations.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

all_correlations["Absolute_Correlation"] = (
    all_correlations["Correlation"].abs()
)

all_correlations = all_correlations.sort_values(
    "Absolute_Correlation",
    ascending=False
).reset_index(drop=True)

display(all_correlations.head(20).round(4))

,Feature_1,Feature_2,Correlation,Absolute_Correlation
0,Open,High,0.9999,0.9999
1,Low,Close,0.9999,0.9999
2,High,Close,0.9999,0.9999
3,Open,Low,0.9998,0.9998
4,High,Low,0.9998,0.9998
5,Open,Close,0.9998,0.9998
6,Employment,Labour_Force,0.9928,0.9928
7,GDP_Real_Millions_CAD,Employment,0.9920,0.9920
8,CPI,Labour_Force,0.9901,0.9901
9,GDP_Real_Millions_CAD,Labour_Force,0.9881,0.9881


In [126]:
stock_target_data = daily_data[
    ["Date", TO_PREDICT] + NUMERIC
].copy()

stock_target_corr = (
    stock_target_data[NUMERIC + [TO_PREDICT]]
    .corr(method="pearson")[TO_PREDICT]
    .drop(TO_PREDICT)
)

stock_target_corr = (
    stock_target_corr
    .to_frame("Correlation_With_Target")
)

stock_target_corr["Absolute_Correlation"] = (
    economic_target_corr["Correlation_With_Target"].abs()
)

stock_target_corr = stock_target_corr.sort_values(
    "Absolute_Correlation",
    ascending=False
)

display(stock_target_corr.round(4))

,Correlation_With_Target,Absolute_Correlation
Policy_Rate_Change_21D,-0.0436,0.0437
USD_CAD,0.0434,0.0434
BCPI_Energy,-0.0387,0.0387
BCPI_Total,-0.0307,0.0307
USD_CAD_Change_21D,0.0196,0.0196
BCPI_Agriculture,-0.0182,0.0182
BCPI_Change_12M,-0.0177,0.0177
Inflation_YoY,-0.0174,0.0174
BCPI_Energy_Change_12M,-0.0169,0.0169
Employment_Growth_12M,-0.0167,0.0167


In [127]:
stock_target_spearman = (
    stock_target_data[NUMERIC + [TO_PREDICT]]
    .corr(method="spearman")[TO_PREDICT]
    .drop(TO_PREDICT)
)

spearman_comparison = pd.DataFrame({
    "Pearson": stock_target_corr[
        "Correlation_With_Target"
    ],
    "Spearman": stock_target_spearman
})

spearman_comparison["Spearman_Absolute"] = (
    spearman_comparison["Spearman"].abs()
)

spearman_comparison = spearman_comparison.sort_values(
    "Spearman_Absolute",
    ascending=False
)

display(spearman_comparison.round(4))

,Pearson,Spearman,Spearman_Absolute
Close_MA_Ratio_20,-0.0660,-0.0485,0.0485
Close_RSI_14,-0.0408,-0.0455,0.0455
Close_MA_Ratio_50,-0.0605,-0.0447,0.0447
MACD_Pct,-0.0748,-0.0440,0.0440
Close_Momentum_10D,-0.0524,-0.0431,0.0431
Close_Momentum_20D,-0.0537,-0.0421,0.0421
MACD_Signal_Pct,-0.0670,-0.0409,0.0409
USD_CAD,0.0434,0.0400,0.0400
Close_MA_Ratio_10,-0.0503,-0.0400,0.0400
Close_Momentum_5D,-0.0428,-0.0332,0.0332


### Stock Market Data Correlation

Pearson correlation was used to examine the relationships between numerical features in the historical stock market dataset.

The analysis helps identify strongly correlated variables, understand relationships between market indicators, and detect potential multicollinearity among features.

Since the dataset contains records from ten selected Canadian stocks, the correlation matrix represents the combined dataset rather than separate correlations for each ticker.

In [22]:
economic_corr = economic_data.corr(method="pearson")

print("Economic Feature Correlation Matrix:")
display(economic_corr.round(2))

Economic Feature Correlation Matrix:


,Policy_Rate,Bond_Yield_10Y,USD_CAD,CPI,Inflation_YoY,GDP_Real_Millions_CAD,GDP_MoM,GDP_YoY,Employment,Labour_Force,...,BCPI_Agriculture,BCPI_Fish,BCPI_Change_12M,BCPI_Energy_Change_12M,BCPI_Metals_Change_12M,Unemployment_Change_12M,Employment_Growth_12M,Yield_Spread,Policy_Rate_Change_21D,USD_CAD_Change_21D
Policy_Rate,1.00,0.58,0.54,0.74,0.52,0.69,0.00,0.14,0.75,0.70,...,0.53,0.61,-0.12,-0.10,-0.15,-0.07,0.27,-0.76,0.05,0.07
Bond_Yield_10Y,0.58,1.00,0.49,-0.46,0.19,-0.54,0.04,0.15,-0.48,-0.52,...,-0.38,-0.28,0.10,0.12,0.13,-0.05,0.20,0.09,0.14,0.10
USD_CAD,0.54,0.49,1.00,0.66,0.01,0.62,-0.11,-0.20,0.64,0.65,...,0.30,0.39,-0.46,-0.41,-0.17,0.22,-0.15,-0.36,-0.35,0.25
CPI,0.74,-0.46,0.66,1.00,0.25,0.99,-0.02,-0.09,0.99,0.99,...,0.79,0.89,-0.06,-0.14,0.02,0.01,0.04,-0.61,-0.02,0.05
Inflation_YoY,0.52,0.19,0.01,0.25,1.00,0.24,0.05,0.50,0.23,0.17,...,0.34,0.38,0.46,0.47,0.19,-0.55,0.54,-0.40,0.37,0.16
GDP_Real_Millions_CAD,0.69,-0.54,0.62,0.99,0.24,1.00,0.01,-0.04,0.99,0.99,...,0.78,0.88,-0.05,-0.12,0.01,-0.05,0.08,-0.65,-0.02,0.09
GDP_MoM,0.00,0.04,-0.11,-0.02,0.05,0.01,1.00,0.21,0.01,0.00,...,0.01,0.02,0.17,0.13,0.10,-0.07,0.14,0.03,0.01,0.04
GDP_YoY,0.14,0.15,-0.20,-0.09,0.50,-0.04,0.21,1.00,-0.03,-0.10,...,0.09,0.09,0.62,0.64,0.28,-0.89,0.89,-0.08,0.15,0.21
Employment,0.75,-0.48,0.64,0.99,0.23,0.99,0.01,-0.03,1.00,0.99,...,0.79,0.87,-0.06,-0.13,0.02,-0.06,0.11,-0.61,-0.05,0.09
Labour_Force,0.70,-0.52,0.65,0.99,0.17,0.99,0.00,-0.10,0.99,1.00,...,0.79,0.86,-0.07,-0.16,0.02,0.02,0.03,-0.57,-0.08,0.05


In [23]:
correlation_threshold = 0.90

upper_triangle = economic_corr.where(
    np.triu(
        np.ones(economic_corr.shape),
        k=1
    ).astype(bool)
)

high_correlations = (
    upper_triangle
    .stack()
    .reset_index()
)

high_correlations.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

high_correlations["Absolute_Correlation"] = (
    high_correlations["Correlation"].abs()
)

high_correlations = (
    high_correlations[
        high_correlations["Absolute_Correlation"] >= correlation_threshold
    ]
    .sort_values("Absolute_Correlation", ascending=False)
    .reset_index(drop=True)
)

print("Highly Correlated Economic Feature Pairs:")
display(high_correlations.round(4))

Highly Correlated Economic Feature Pairs:


,Feature_1,Feature_2,Correlation,Absolute_Correlation
0,Employment,Labour_Force,0.9930,0.9930
1,GDP_Real_Millions_CAD,Employment,0.9921,0.9921
2,CPI,Labour_Force,0.9901,0.9901
3,GDP_Real_Millions_CAD,Labour_Force,0.9881,0.9881
4,CPI,Employment,0.9855,0.9855
5,CPI,GDP_Real_Millions_CAD,0.9853,0.9853
6,BCPI_NonEnergy,BCPI_Metals,0.9704,0.9704
7,BCPI_Change_12M,BCPI_Energy_Change_12M,0.9520,0.9520
8,BCPI_Total,BCPI_Energy,0.9452,0.9452
9,BCPI_NonEnergy,BCPI_Agriculture,0.9361,0.9361


In [24]:
all_correlations = (
    upper_triangle
    .stack()
    .reset_index()
)

all_correlations.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

all_correlations["Absolute_Correlation"] = (
    all_correlations["Correlation"].abs()
)

all_correlations = all_correlations.sort_values(
    "Absolute_Correlation",
    ascending=False
).reset_index(drop=True)

display(all_correlations.head(20).round(4))

,Feature_1,Feature_2,Correlation,Absolute_Correlation
0,Employment,Labour_Force,0.9930,0.9930
1,GDP_Real_Millions_CAD,Employment,0.9921,0.9921
2,CPI,Labour_Force,0.9901,0.9901
3,GDP_Real_Millions_CAD,Labour_Force,0.9881,0.9881
4,CPI,Employment,0.9855,0.9855
5,CPI,GDP_Real_Millions_CAD,0.9853,0.9853
6,BCPI_NonEnergy,BCPI_Metals,0.9704,0.9704
7,BCPI_Change_12M,BCPI_Energy_Change_12M,0.9520,0.9520
8,BCPI_Total,BCPI_Energy,0.9452,0.9452
9,BCPI_NonEnergy,BCPI_Agriculture,0.9361,0.9361


In [22]:
level_features = [
    "CPI",
    "GDP_Real_Millions_CAD",
    "Employment",
    "Labour_Force",
    "BCPI_Total",
    "BCPI_NonEnergy",
    "BCPI_Energy",
    "BCPI_Metals",
    "BCPI_Agriculture"
]

transformed_features = [
    "Inflation_YoY",
    "GDP_MoM",
    "GDP_YoY",
    "Employment_Growth_12M",
    "BCPI_Change_12M",
    "BCPI_Energy_Change_12M",
    "BCPI_Metals_Change_12M"
]

correlation_data = economic_data[
    level_features + transformed_features
].corr(method="pearson")

display(correlation_data.round(3))

,CPI,GDP_Real_Millions_CAD,Employment,Labour_Force,BCPI_Total,BCPI_NonEnergy,BCPI_Energy,BCPI_Metals,BCPI_Agriculture,Inflation_YoY,GDP_MoM,GDP_YoY,Employment_Growth_12M,BCPI_Change_12M,BCPI_Energy_Change_12M,BCPI_Metals_Change_12M
CPI,1.000,0.985,0.985,0.990,0.536,0.866,0.285,0.844,0.790,0.250,-0.017,-0.090,0.036,-0.064,-0.137,0.017
GDP_Real_Millions_CAD,0.985,1.000,0.992,0.988,0.524,0.850,0.286,0.822,0.775,0.236,0.012,-0.035,0.082,-0.048,-0.125,0.013
Employment,0.985,0.992,1.000,0.993,0.562,0.861,0.329,0.843,0.795,0.232,0.007,-0.033,0.107,-0.056,-0.134,0.016
Labour_Force,0.990,0.988,0.993,1.000,0.545,0.860,0.306,0.851,0.792,0.172,0.004,-0.099,0.032,-0.072,-0.155,0.016
BCPI_Total,0.536,0.524,0.562,0.545,1.000,0.823,0.945,0.814,0.845,0.415,0.078,0.188,0.228,0.329,0.226,0.216
BCPI_NonEnergy,0.866,0.850,0.861,0.860,0.823,1.000,0.600,0.970,0.936,0.384,0.027,0.098,0.182,0.186,0.064,0.231
BCPI_Energy,0.285,0.286,0.329,0.306,0.945,0.600,1.000,0.607,0.669,0.346,0.085,0.192,0.202,0.354,0.280,0.154
BCPI_Metals,0.844,0.822,0.843,0.851,0.814,0.970,0.607,1.000,0.900,0.276,0.010,-0.022,0.070,0.113,-0.009,0.255
BCPI_Agriculture,0.790,0.775,0.795,0.792,0.845,0.936,0.669,0.900,1.000,0.341,0.010,0.091,0.179,0.070,-0.021,0.008
Inflation_YoY,0.250,0.236,0.232,0.172,0.415,0.384,0.346,0.276,0.341,1.000,0.048,0.500,0.541,0.463,0.468,0.194


In [26]:
economic_target_data = daily_data[
    ["Date", TO_PREDICT] + ECONOMIC
].copy()

economic_target_corr = (
    economic_target_data[ECONOMIC + [TO_PREDICT]]
    .corr(method="pearson")[TO_PREDICT]
    .drop(TO_PREDICT)
)

economic_target_corr = (
    economic_target_corr
    .to_frame("Correlation_With_Target")
)

economic_target_corr["Absolute_Correlation"] = (
    economic_target_corr["Correlation_With_Target"].abs()
)

economic_target_corr = economic_target_corr.sort_values(
    "Absolute_Correlation",
    ascending=False
)

display(economic_target_corr.round(4))

,Correlation_With_Target,Absolute_Correlation
Policy_Rate_Change_21D,-0.0437,0.0437
USD_CAD,0.0434,0.0434
BCPI_Energy,-0.0387,0.0387
BCPI_Total,-0.0307,0.0307
USD_CAD_Change_21D,0.0196,0.0196
BCPI_Agriculture,-0.0182,0.0182
BCPI_Change_12M,-0.0177,0.0177
Inflation_YoY,-0.0174,0.0174
BCPI_Energy_Change_12M,-0.0169,0.0169
Employment_Growth_12M,-0.0167,0.0167


In [27]:
economic_target_spearman = (
    economic_target_data[ECONOMIC + [TO_PREDICT]]
    .corr(method="spearman")[TO_PREDICT]
    .drop(TO_PREDICT)
)

spearman_comparison = pd.DataFrame({
    "Pearson": economic_target_corr[
        "Correlation_With_Target"
    ],
    "Spearman": economic_target_spearman
})

spearman_comparison["Spearman_Absolute"] = (
    spearman_comparison["Spearman"].abs()
)

spearman_comparison = spearman_comparison.sort_values(
    "Spearman_Absolute",
    ascending=False
)

display(spearman_comparison.round(4))

,Pearson,Spearman,Spearman_Absolute
USD_CAD,0.0434,0.0400,0.0400
Policy_Rate_Change_21D,-0.0437,-0.0293,0.0293
BCPI_Energy,-0.0387,-0.0219,0.0219
USD_CAD_Change_21D,0.0196,-0.0188,0.0188
BCPI_Total,-0.0307,-0.0173,0.0173
Employment_Growth_12M,-0.0167,-0.0169,0.0169
Participation_Rate,-0.0130,-0.0155,0.0155
Unemployment_Rate,0.0144,0.0146,0.0146
BCPI_Agriculture,-0.0182,-0.0137,0.0137
GDP_YoY,-0.0165,-0.0118,0.0118


### Economic Feature Correlation

Pearson correlation was applied to examine relationships among the economic indicators used in the Market Regime Prediction model.

The analysis helps identify relationships between macroeconomic variables, detect potential multicollinearity, and better understand how economic indicators move in relation to one another.

These findings support economic feature analysis and help inform feature selection for the market regime classification task.

### Future Outperformance Target (`Future_Outperformance_5D`)

The target variable identifies whether a stock outperforms the median return of the selected Canadian stocks over the following five trading days.

First, the five-day future return is calculated for each stock using its Adjusted Closing Price:

\[
Future\ Return_{5D} = \left(\frac{AdjClose_{t+5}}{AdjClose_t}-1\right)\times100
\]

Next, each stock's future return is compared with the cross-sectional median future return of all selected stocks on the same date.

The target is assigned as follows:

- **1 (Outperformer):** The stock's five-day future return is greater than the median return.
- **0 (Underperformer):** The stock's five-day future return is less than or equal to the median return.
- **NaN:** Assigned when the five-day future return cannot be calculated, such as at the end of the dataset.

This binary target enables the classification model to estimate the probability that a selected stock will outperform its peers, rather than predict its exact future price or return.

In [14]:
daily_data["Future_Outperformance_5D"] = (
    daily_data["Future_Return_5D"]
    >
    daily_data.groupby("Date")["Future_Return_5D"]
    .transform("median")
).astype("float")

daily_data.loc[
    daily_data["Future_Return_5D"].isna(),
    "Future_Outperformance_5D"
] = float("nan")

In [15]:
print(
    daily_data["Future_Outperformance_5D"]
    .value_counts(dropna=False)
)

print(
    daily_data["Future_Outperformance_5D"]
    .value_counts(normalize=True)
)

Future_Outperformance_5D
0.0    33696
1.0    29828
NaN       50
Name: count, dtype: int64
Future_Outperformance_5D
0.0    0.530445
1.0    0.469555
Name: proportion, dtype: float64


In [16]:
TARGET = "Future_Outperformance_5D"

model_data = daily_data.dropna(subset=[TARGET]).copy()

X = model_data[FEATURES]
y = model_data[TARGET].astype(int)

train_mask = model_data["Date"] <= pd.Timestamp("2018-10-02")

val_mask = (
    (model_data["Date"] >= pd.Timestamp("2018-10-11")) &
    (model_data["Date"] <= pd.Timestamp("2022-10-12"))
)

test_mask = (
    (model_data["Date"] >= pd.Timestamp("2022-10-20")) &
    (model_data["Date"] <= pd.Timestamp("2026-10-30"))
)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (43254, 51) (43254,)
Validation: (10050, 51) (10050,)
Test: (10120, 51) (10120,)


# Stock Ranking Evaluation

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [29]:
# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            DUMMIES
        ),
        (
            "numeric",
            SimpleImputer(strategy="median"),
            NUMERIC
        )
    ]
)

# Decision Tree
decision_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42
    ))
])

# Random Forest
random_forest = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        max_features=0.7,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ))
])

# XGDB
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

print("all models are ready.")

all models are ready.


In [30]:
models = {
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
    "XGBoost": xgb_model
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained successfully.")

Decision Tree trained successfully.
Random Forest trained successfully.
XGBoost trained successfully.


In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = []

for name, model in models.items():
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1": f1_score(y_val, y_pred),
        "ROC_AUC": roc_auc_score(y_val, y_prob)
    })

results_df = pd.DataFrame(results).sort_values(
    "ROC_AUC", ascending=False
)

results_df.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
1,Random Forest,0.5096,0.5064,0.7528,0.6055,0.5137
0,Decision Tree,0.5042,0.5024,0.8872,0.6415,0.5123
2,XGBoost,0.5109,0.5117,0.4784,0.4945,0.5117


In [32]:
from scipy.stats import spearmanr

ranking_results = []

val_data = model_data.loc[val_mask].copy()

for name, model in models.items():

    val_data["Predicted_Probability"] = (
        model.predict_proba(X_val)[:, 1]
    )

    daily_correlations = (
        val_data.groupby("Date")
        .apply(
            lambda group: spearmanr(
                group["Predicted_Probability"],
                group["Future_Return_5D"]
            ).statistic,
            include_groups=False
        )
        .dropna()
    )

    ranking_results.append({
        "Model": name,
        "Mean_Spearman": daily_correlations.mean(),
        "Median_Spearman": daily_correlations.median(),
        "Positive_Rank_Correlation": (
            daily_correlations > 0
        ).mean(),
        "Valid_Days": len(daily_correlations)
    })

ranking_df = pd.DataFrame(ranking_results)

ranking_df.sort_values(
    "Mean_Spearman", ascending=False
).round(4)

,Model,Mean_Spearman,Median_Spearman,Positive_Rank_Correlation,Valid_Days
0,Decision Tree,0.0368,0.0605,0.5403,1005
2,XGBoost,0.0276,0.0424,0.5433,1005
1,Random Forest,0.0193,0.0182,0.5154,1005


In [33]:
top3_results = []

val_data = model_data.loc[val_mask].copy()

for name, model in models.items():

    val_data["Predicted_Probability"] = (
        model.predict_proba(X_val)[:, 1]
    )

    daily_scores = []

    for date, group in val_data.groupby("Date"):

        if len(group) < 6:
            continue

        predicted_top3 = set(
            group.nlargest(3, "Predicted_Probability")["Ticker"]
        )

        actual_top3 = set(
            group.nlargest(3, "Future_Return_5D")["Ticker"]
        )

        daily_scores.append(
            len(predicted_top3 & actual_top3) / 3
        )

    top3_results.append({
        "Model": name,
        "Precision@3": np.mean(daily_scores),
        "Valid_Days": len(daily_scores)
    })

top3_df = pd.DataFrame(top3_results)

top3_df.sort_values(
    "Precision@3", ascending=False
).round(4)

,Model,Precision@3,Valid_Days
0,Decision Tree,0.3473,1005
1,Random Forest,0.3469,1005
2,XGBoost,0.3449,1005


In [41]:
from sklearn.model_selection import ParameterGrid
from sklearn.base import clone

dt_params = {
    "model__max_depth": [4, 6, 8, 10, 12],
    "model__min_samples_leaf": [20, 50, 100],
    "model__min_samples_split": [50, 100, 200],
    "model__criterion": ["gini", "entropy"]
}

dt_results = []

for params in ParameterGrid(dt_params):

    model = clone(decision_tree)
    model.set_params(**params)
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_val)[:, 1]

    temp = model_data.loc[val_mask, ["Date", "Ticker", "Future_Return_5D"]].copy()
    temp["Predicted_Probability"] = probabilities

    daily_scores = []

    for date, group in temp.groupby("Date"):

        if len(group) < 6:
            continue

        predicted_top3 = set(
            group.nlargest(3, "Predicted_Probability")["Ticker"]
        )

        actual_top3 = set(
            group.nlargest(3, "Future_Return_5D")["Ticker"]
        )

        daily_scores.append(
            len(predicted_top3 & actual_top3) / 3
        )

    dt_results.append({
        **params,
        "Precision@3": np.mean(daily_scores)
    })

dt_tuning_df = pd.DataFrame(dt_results).sort_values(
    "Precision@3", ascending=False
)

dt_tuning_df.head(10).round(4)

,model__criterion,model__max_depth,model__min_samples_leaf,model__min_samples_split,Precision@3
28,gini,10,20,100,0.3602
37,gini,12,20,100,0.3566
70,entropy,8,100,100,0.3546
71,entropy,8,100,200,0.3546
69,entropy,8,100,50,0.3546
79,entropy,10,100,100,0.3476
80,entropy,10,100,200,0.3476
78,entropy,10,100,50,0.3476
33,gini,10,100,50,0.3476
35,gini,10,100,200,0.3476


In [42]:
print(daily_data.shape)
print(daily_data.index[:5].tolist())
print(daily_data[["Date", "Ticker"]].head())

(63574, 69)
[0, 1, 2, 3, 4]
        Date  Ticker
0 2000-01-04  ABX.TO
1 2000-01-05  ABX.TO
2 2000-01-06  ABX.TO
3 2000-01-07  ABX.TO
4 2000-01-10  ABX.TO


In [43]:
# Display all columns
for i, column in enumerate(daily_data.columns, start=1):
    print(f"{i:02d}. {column}")

01. Date
02. Ticker
03. Open
04. High
05. Low
06. Close
07. Adj Close
08. Volume
09. Dividends
10. Stock Splits
11. Bank_Availability_Date
12. Policy_Rate
13. Bond_Yield_10Y
14. USD_CAD
15. Macro_Reference_Date
16. CPI
17. Inflation_YoY
18. GDP_Real_Millions_CAD
19. GDP_MoM
20. GDP_YoY
21. Employment
22. Labour_Force
23. Participation_Rate
24. Unemployment_Rate
25. BCPI_Total
26. BCPI_NonEnergy
27. BCPI_Energy
28. BCPI_Metals
29. BCPI_Forestry
30. BCPI_Agriculture
31. BCPI_Fish
32. Macro_Availability_Date
33. DayOfWeek
34. Month
35. Quarter
36. DayOfWeek_Sin
37. DayOfWeek_Cos
38. Month_Sin
39. Month_Cos
40. Macro_Feature_Availability_Date
41. BCPI_Change_12M
42. BCPI_Energy_Change_12M
43. BCPI_Metals_Change_12M
44. Unemployment_Change_12M
45. Employment_Growth_12M
46. Yield_Spread
47. Policy_Rate_Change_21D
48. USD_CAD_Change_21D
49. Future_Return_5D
50. Close_Daily_Return
51. Close_Return_Lag_1
52. Close_Return_Lag_5
53. Close_Return_Lag_10
54. Close_Return_Lag_20
55. Close_Volatility

### Stock Ranking Evaluation

Three classification models (Decision Tree, Random Forest, and XGBoost) were evaluated using Spearman Rank Correlation and Precision@3 to assess their ability to identify stocks with higher future outperformance probabilities.

Decision Tree achieved the best overall results, with a mean Spearman correlation of 0.0368 and Precision@3 of 34.73%. Random Forest achieved a Precision@3 of 34.69%, while XGBoost achieved 34.49%.

All models slightly outperformed the random selection baseline of 30%. However, the low Spearman correlation values indicate limited ranking ability.

Based on these results, Decision Tree was selected as the preferred candidate for the trading strategy, while acknowledging the relatively modest predictive performance.

# Stock Classification – Model Development and Prediction

In [17]:
# Original stock features
STOCK_FEATURES = TECHNICAL + ECONOMIC + CALENDAR + ["Ticker"]

STOCK_TARGET = "Future_Outperformance_5D"

stock_data = (
    daily_data
    .dropna(subset=[STOCK_TARGET])
    .sort_values(["Date", "Ticker"])
    .copy()
)

unique_dates = sorted(stock_data["Date"].unique())

n_dates = len(unique_dates)
train_end = int(n_dates * 0.60)
val_end = int(n_dates * 0.80)

train_dates = unique_dates[:train_end - 5]
val_dates = unique_dates[train_end:val_end - 5]
test_dates = unique_dates[val_end:]

train_mask = stock_data["Date"].isin(train_dates)
val_mask = stock_data["Date"].isin(val_dates)
test_mask = stock_data["Date"].isin(test_dates)

X_stock_train = stock_data.loc[train_mask, STOCK_FEATURES]
y_stock_train = stock_data.loc[train_mask, STOCK_TARGET]

X_stock_val = stock_data.loc[val_mask, STOCK_FEATURES]
y_stock_val = stock_data.loc[val_mask, STOCK_TARGET]

X_stock_test = stock_data.loc[test_mask, STOCK_FEATURES]
y_stock_test = stock_data.loc[test_mask, STOCK_TARGET]

print("Train:", X_stock_train.shape, y_stock_train.shape)
print("Validation:", X_stock_val.shape, y_stock_val.shape)
print("Test:", X_stock_test.shape, y_stock_test.shape)

Train: (36514, 51) (36514,)
Validation: (13430, 51) (13430,)
Test: (13480, 51) (13480,)


In [18]:
for name, mask in [
    ("Stock Train", train_mask),
    ("Stock Validation", val_mask),
    ("Stock Test", test_mask)
]:
    dates = stock_data.loc[mask, "Date"]
    print(f"{name}: {dates.min().date()} -> {dates.max().date()}")

Stock Train: 2000-01-04 -> 2016-01-26
Stock Validation: 2016-02-03 -> 2021-06-09
Stock Test: 2021-06-17 -> 2026-10-30


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# Preprocessing
stock_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=True
            ),
            TECHNICAL + ECONOMIC + CALENDAR
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ["Ticker"]
        )
    ]
)

# Best tuned Decision Tree
stock_model = Pipeline([
    ("preprocessor", stock_preprocessor),
    ("model", DecisionTreeClassifier(
        criterion="gini",
        max_depth=10,
        min_samples_leaf=20,
        min_samples_split=100,
        class_weight="balanced",
        random_state=42
    ))
])

# Train only
stock_model.fit(X_stock_train, y_stock_train)

# Probability of outperforming
positive_class_index = list(
    stock_model.named_steps["model"].classes_
).index(1)

val_probabilities = stock_model.predict_proba(X_stock_val)[:, positive_class_index]
test_probabilities = stock_model.predict_proba(X_stock_test)[:, positive_class_index]

# Save predictions with their dates and tickers
stock_val_predictions = stock_data.loc[
    val_mask, ["Date", "Ticker", STOCK_TARGET]
].copy()

stock_val_predictions["Stock_Outperformance_Prob"] = val_probabilities

stock_test_predictions = stock_data.loc[
    test_mask, ["Date", "Ticker", STOCK_TARGET]
].copy()

stock_test_predictions["Stock_Outperformance_Prob"] = test_probabilities

print("Validation predictions:", stock_val_predictions.shape)
print("Test predictions:", stock_test_predictions.shape)

print(stock_val_predictions.head())

Validation predictions: (13430, 4)
Test predictions: (13480, 4)
            Date  Ticker  Future_Outperformance_5D  Stock_Outperformance_Prob
4043  2016-02-03  ABX.TO                       1.0                   0.123858
10787 2016-02-03  BMO.TO                       1.0                   0.350613
17531 2016-02-03   BN.TO                       1.0                   0.417439
24275 2016-02-03  CNQ.TO                       0.0                   0.437419
31019 2016-02-03  CNR.TO                       1.0                   0.350613


### Stock Classification Model Development

A Decision Tree Classifier was developed using technical, economic, and calendar features, with the stock ticker included as a categorical variable. The target variable was `Future_Outperformance_5D`, representing whether a stock outperforms the median five-day future return of the selected stocks.

The dataset was divided chronologically into training, validation, and test sets using a 60/20/20 split, with a five-trading-day gap to reduce target overlap. Numerical missing values were handled using median imputation with missing-value indicators, while the ticker was encoded using One-Hot Encoding.

After model comparison and hyperparameter tuning, the selected Decision Tree configuration used a maximum depth of 10, a minimum of 20 samples per leaf, and a minimum of 100 samples required for splitting, with balanced class weights.

The model was trained exclusively on the training set and generated outperformance probabilities for the validation and test sets. These probabilities were then used to rank stocks and support the subsequent trading strategy simulations.

# Strategy 1: Top 3 Stocks – 5 Trading Days

## Strategy 1: Top 3 Stocks – 5 Trading Days

This strategy uses the Decision Tree stock classification model to select the three stocks with the highest predicted outperformance probabilities each trading day. It operates independently of the Market Regime Prediction model.

Each selected stock receives a fixed investment of $100 and is held for five trading days, with a round-trip transaction fee of 0.2%. When predicted probabilities are tied, `Close_Momentum_5D` is used as a secondary ranking criterion.

The simulation generated 4,044 trades, achieving a total net profit of $184.94 and a win rate of 53.09%. The maximum number of concurrent open positions was 15, requiring an estimated capital of $1,500.

Over the testing period from June 2021 to November 2026, the strategy achieved a total return of 12.33% and a CAGR of 2.18%.

These results establish the baseline performance for comparison with the second strategy, which incorporates market regime predictions.

### Configurable Parameters

The strategy uses four adjustable parameters: `INITIAL_INVESTMENT` ($100 per stock), `TOP_N` (3 selected stocks), `HOLDING_DAYS` (5 trading days), and `ROUND_TRIP_FEE` (0.2%).

These values represent the default configuration rather than fixed constraints and can be modified to experiment with different trading scenarios. Any change to the holding period also requires updating the corresponding future return calculations and exit dates.

In [321]:
def strategy_stock_trading(
    stock_test_predictions,
    daily_data,
    INITIAL_INVESTMENT = 100,
    TOP_N = 3,
     HOLDING_DAYS = 5,
     ROUND_TRIP_FEE = 0.002
     ):
    
    # ------------------------------------------
    # 1. Prepare future returns and exit dates
    # ------------------------------------------

    exit_data = daily_data[
        ["Date", "Ticker", "Future_Return_5D",
        "Close_Momentum_5D"]
    ].sort_values(["Ticker", "Date"]).copy()

    exit_data["Exit_Date"] = (
        exit_data.groupby("Ticker")["Date"].shift(-HOLDING_DAYS)
    )

    # ------------------------------------------
    # 2. Merge predictions with actual returns
    # ------------------------------------------

    simulation_data = stock_test_predictions.merge(
        exit_data,
        on=["Date", "Ticker"],
        how="inner",
        validate="one_to_one"
    )

    simulation_data = simulation_data.dropna(
        subset=[
            "Stock_Outperformance_Prob",
            "Future_Return_5D",
            "Exit_Date"
        ]
    )

    # ------------------------------------------
    # 3. Select Top 3 stocks per trading day
    # ------------------------------------------

    selected_trades = (
        simulation_data
        .sort_values(
            ["Date", "Stock_Outperformance_Prob", "Close_Momentum_5D"],
            ascending=[True, False, False]
        )
        .groupby("Date")
        .head(TOP_N)
        .copy()
    )

    # ------------------------------------------
    # 4. Calculate financial results
    # ------------------------------------------

    selected_trades["Investment"] = INITIAL_INVESTMENT

    selected_trades["Gross_Profit"] = (
        INITIAL_INVESTMENT *
        selected_trades["Future_Return_5D"] / 100
    )

    selected_trades["Fees"] = (
        INITIAL_INVESTMENT * ROUND_TRIP_FEE
    )

    selected_trades["Net_Profit"] = (
        selected_trades["Gross_Profit"] -
        selected_trades["Fees"]
    )

    # ------------------------------------------
    # 5. Calculate investment statistics
    # ------------------------------------------

    total_trades = len(selected_trades)

    gross_profit = selected_trades["Gross_Profit"].sum()
    total_fees = selected_trades["Fees"].sum()
    net_profit = selected_trades["Net_Profit"].sum()

    average_profit_per_trade = (
        net_profit / total_trades
        if total_trades > 0 else 0
    )

    win_rate = (
        (selected_trades["Net_Profit"] > 0).mean()
        if total_trades > 0 else 0
    )

    # ------------------------------------------
    # 6. Calculate required capital
    # ------------------------------------------

    events = pd.concat([
        selected_trades[["Date"]].rename(
            columns={"Date": "Event_Date"}
        ).assign(Change=1),

        selected_trades[["Exit_Date"]].rename(
            columns={"Exit_Date": "Event_Date"}
        ).assign(Change=-1)
    ])

    daily_positions = (
        events.groupby("Event_Date")["Change"]
        .sum()
        .sort_index()
        .cumsum()
    )

    selected_trades["Entry_Date"] = selected_trades["Date"]

    selected_trades["Entry_Date"] = pd.to_datetime(
        selected_trades["Entry_Date"]
    )
    
    max_open_positions = daily_positions.max()

    required_capital = (
        max_open_positions * INITIAL_INVESTMENT
    )

    # ------------------------------------------
    # 7. Calculate daily portfolio equity
    # ------------------------------------------

    daily_profit = (
        selected_trades.groupby("Exit_Date")["Net_Profit"]
        .sum()
    )

    trading_dates = (
        daily_data.loc[
            (daily_data["Date"] >= selected_trades["Date"].min()) &
            (daily_data["Date"] <= selected_trades["Exit_Date"].max()),
            "Date"
        ]
        .drop_duplicates()
        .sort_values()
    )

    equity = (
        daily_profit
        .reindex(trading_dates, fill_value=0)
        .cumsum()
        + required_capital
    )

    equity = equity.rename("Portfolio_Value")

    # ------------------------------------------
    # 8. Calculate CAGR and Maximum Drawdown
    # ------------------------------------------

    if len(equity) > 1:
        duration_days = (
            equity.index[-1] - equity.index[0]
        ).days

        final_value = equity.iloc[-1]

        if required_capital > 0 and final_value > 0:
            cagr = (
                (final_value / required_capital)
                ** (365.25 / duration_days)
            ) - 1
        else:
            cagr = np.nan

        drawdown = equity / equity.cummax() - 1
        max_drawdown = drawdown.min()

    else:
        final_value = required_capital
        cagr = np.nan
        max_drawdown = np.nan

    start_date = selected_trades["Date"].min()
    end_date = selected_trades["Exit_Date"].max()

    years = (end_date - start_date).days / 365.25

    total_return = (
        net_profit / required_capital * 100
        if required_capital > 0 else np.nan
        )

    cagr = (
        ((1 + total_return / 100) ** (1 / years) - 1) * 100
        if years > 0 and total_return > -100
        else np.nan
        )

    
    # ------------------------------------------
    # 9. Final results
    # ------------------------------------------

    
    strategy_results = pd.DataFrame([{
        "Strategy": "Top 3 Stocks",
        "Total_Trades": total_trades,
        "Gross_Profit": gross_profit,
        "Total_Fees": total_fees,
        "Net_Profit": net_profit,
        "Average_Profit_Per_Trade": average_profit_per_trade,
        "Win_Rate": win_rate,
        "Max_Open_Positions": max_open_positions,
        "Required_Capital": required_capital,
        "Final_Portfolio_Value": final_value,
        "CAGR": cagr,
        "Maximum_Drawdown": max_drawdown,
        "Entry_Date": start_date,
        "Exit_Date": end_date,
        "Testing_Period_Years": years,
        "Total_Return": total_return,
    }])

    # ------------------------------------------
    # 10. Latest Selected Trades
    # ------------------------------------------

    latest_trades = (
        selected_trades[
            selected_trades["Date"].isin(
                simulation_data["Date"]
                .drop_duplicates()
                .sort_values()
                .head(5)
            )
        ][
            [
                "Date",
                "Entry_Date",
                "Exit_Date",
                "Ticker",
                "Stock_Outperformance_Prob",
                "Close_Momentum_5D",
                "Future_Return_5D",
                "Net_Profit",
                "Gross_Profit",
                "Fees",
                "Investment"
            ]
        ]
        .sort_values(["Date", "Stock_Outperformance_Prob"], ascending=[True, False])
        .reset_index(drop=True)
    )

    # ------------------------------------------
    # 11. Return Results
    # ------------------------------------------

    return strategy_results, selected_trades, latest_trades, simulation_data


In [322]:
strategy_results, strategy1_trades, latest_trades, simulation_data = strategy_stock_trading(
    stock_test_predictions,
    daily_data
)

display(strategy_results)
# display(strategy1_trades)

,Strategy,Total_Trades,Gross_Profit,Total_Fees,Net_Profit,Average_Profit_Per_Trade,Win_Rate,Max_Open_Positions,Required_Capital,Final_Portfolio_Value,CAGR,Maximum_Drawdown,Entry_Date,Exit_Date,Testing_Period_Years,Total_Return
0,Top 3 Stocks,4044,629.152615,808.8,-179.647385,-0.044423,0.541296,15,1500,1320.352615,-2.339753,-0.361421,2021-06-17,2026-11-06,5.38809,-11.976492


In [266]:
strategy1_trades[
    strategy1_trades["Date"].isin(
        simulation_data["Date"].drop_duplicates().sort_values().head(5)
    )
][
    [
        "Date",
        "Ticker",
        "Stock_Outperformance_Prob",
        "Close_Momentum_5D",
        "Future_Return_5D",
        "Net_Profit"
    ]
]

,Date,Ticker,Stock_Outperformance_Prob,Close_Momentum_5D,Future_Return_5D,Net_Profit
5,2021-06-17,ENB.TO,0.811208,3.974198,-1.140701,-1.340701
6,2021-06-17,RY.TO,0.811208,1.908585,-1.078084,-1.278084
2,2021-06-17,BN.TO,0.811208,0.977956,4.497700,4.297700
15,2021-06-18,ENB.TO,0.811208,2.403149,0.101136,-0.098864
12,2021-06-18,BN.TO,0.811208,0.314777,5.962010,5.762010
11,2021-06-18,BMO.TO,0.811208,0.007881,1.355057,1.155057
28,2021-06-21,SU.TO,0.704629,-1.077375,-3.102306,-3.302306
23,2021-06-21,CNQ.TO,0.704629,-1.405663,-1.537091,-1.737091
22,2021-06-21,BN.TO,0.529304,1.993021,4.445536,4.245536
32,2021-06-22,BN.TO,0.811208,1.376150,4.751103,4.551103


In [232]:
trade_dates = pd.DatetimeIndex(
    sorted(simulation_data["Date"].unique())
)

active_positions = pd.Series(
    {
        date: (
            (strategy1_trades["Date"] <= date) &
            (strategy1_trades["Exit_Date"] > date)
        ).sum()
        for date in trade_dates
    }
)

max_open_positions = active_positions.max()

required_capital = max_open_positions * INITIAL_INVESTMENT

print("Maximum Open Positions:", max_open_positions)
print("Required Capital ($):", required_capital)

Maximum Open Positions: 15
Required Capital ($): 1500


In [242]:
# Inspect probability ties on the first test date

first_date = strategy1_trades["Date"].min()

display(
    simulation_data[
        simulation_data["Date"] == first_date
    ][
        ["Ticker", "Stock_Outperformance_Prob"]
    ].sort_values(
        "Stock_Outperformance_Prob",
        ascending=False
    )
)

,Ticker,Stock_Outperformance_Prob
1,BMO.TO,0.811208
2,BN.TO,0.811208
9,TD.TO,0.811208
4,CNR.TO,0.811208
5,ENB.TO,0.811208
6,RY.TO,0.811208
8,SU.TO,0.661994
3,CNQ.TO,0.661994
0,ABX.TO,0.627006
7,SHOP.TO,0.123858


In [27]:
daily_ties = (
    simulation_data
    .groupby("Date")["Stock_Outperformance_Prob"]
    .agg(
        Unique_Probabilities="nunique",
        Max_Probability="max"
    )
)

print("Days with ties at maximum probability:")
print(
    (
        simulation_data
        .groupby("Date")
        .apply(
            lambda x: (
                x["Stock_Outperformance_Prob"]
                == x["Stock_Outperformance_Prob"].max()
            ).sum()
        ) > TOP_N
    ).sum()
)

print("Total test days:", len(daily_ties))

Days with ties at maximum probability:
569
Total test days: 1348


In [28]:
first_day = simulation_data[
    simulation_data["Date"] == simulation_data["Date"].min()
]

print(
    first_day[
        ["Ticker", "Stock_Outperformance_Prob", "Close_Momentum_5D"]
    ].sort_values(
        ["Stock_Outperformance_Prob", "Close_Momentum_5D"],
        ascending=False
    )
)

    Ticker  Stock_Outperformance_Prob  Close_Momentum_5D
5   ENB.TO                   0.811208           3.974198
6    RY.TO                   0.811208           1.908585
2    BN.TO                   0.811208           0.977956
1   BMO.TO                   0.811208           0.693190
9    TD.TO                   0.811208           0.551212
4   CNR.TO                   0.811208          -1.747807
8    SU.TO                   0.661994          -1.657825
3   CNQ.TO                   0.661994          -3.197677
0   ABX.TO                   0.627006          -9.417512
7  SHOP.TO                   0.123858          19.873438


# Vectorized Simulation

In [35]:
def vectorized_simulation(
    trades_df,
    initial_capital=1500,
    round_trip_fee=0.002
):
    trades = trades_df.copy()

    trades["Entry_Date"] = pd.to_datetime(trades["Entry_Date"]) 
    trades["Exit_Date"] = pd.to_datetime(trades["Exit_Date"])

    # Convert percentage return to decimal
    returns = trades["Future_Return_5D"] / 100

    trades["Gross_Profit"] = trades["Investment"] * returns
    trades["Fees"] = trades["Investment"] * round_trip_fee
    trades["Net_Profit"] = trades["Gross_Profit"] - trades["Fees"]

    trades["Net_Return"] = (
        trades["Net_Profit"] / trades["Investment"]
    )

    summary = {
        "Total Trades": len(trades),
        "Total Investment": trades["Investment"].sum(),
        "Gross Profit": trades["Gross_Profit"].sum(),
        "Fees": trades["Fees"].sum(),
        "Net Profit": trades["Net_Profit"].sum(),
        "Win Rate (%)": (trades["Net_Profit"] > 0).mean() * 100,
        "Average Net Profit": trades["Net_Profit"].mean(),
        "Initial Capital": initial_capital,
        "Return on Initial Capital (%)": (
            trades["Net_Profit"].sum() / initial_capital * 100
        )
    }

    return trades, pd.Series(summary)

In [267]:
vectorized_trades, vectorized_summary = vectorized_simulation(
    strategy1_trades,
    initial_capital=1500
)

display(vectorized_summary)

Total Trades                       4044.000000
Total Investment                 404400.000000
Gross Profit                        629.152615
Fees                                808.800000
Net Profit                         -179.647385
Win Rate (%)                         54.129575
Average Net Profit                   -0.044423
Initial Capital                    1500.000000
Return on Initial Capital (%)       -11.976492
dtype: float64

# Exact Simulation

In [149]:
def exact_simulation(
    trades_df,
    daily_data,
    initial_capital=1500,
    max_open_positions=15,
    position_fraction=1 / 15,
    round_trip_fee=0.002,
    trading_dates=None
):
    trades = trades_df.copy()
    prices_data = daily_data.copy()

    # 1. Prepare dates
    trades["Entry_Date"] = pd.to_datetime(trades["Entry_Date"])
    trades["Exit_Date"] = pd.to_datetime(trades["Exit_Date"])

    prices_data["Date"] = pd.to_datetime(prices_data["Date"])

    trades = trades.sort_values(
        ["Entry_Date", "Ticker"]
    ).reset_index(drop=True)

    prices_data = prices_data[
        ["Date", "Ticker", "Adj Close"]
    ].drop_duplicates(
        ["Date", "Ticker"]
    )

    # 2. Build daily price matrix
    prices = prices_data.pivot(
        index="Date",
        columns="Ticker",
        values="Adj Close"
    ).sort_index()

    # 3. Build complete trading calendar
    if trading_dates is not None:
        dates = sorted(
            set(pd.to_datetime(trading_dates))
        )
    else:
        start_date = trades["Entry_Date"].min()
        end_date = trades["Exit_Date"].max()

        dates = sorted(
            prices.index[
                (prices.index >= start_date) &
                (prices.index <= end_date)
            ].tolist()
        )

    dates = sorted(
        set(dates)
        .union(trades["Entry_Date"])
        .union(trades["Exit_Date"])
    )

    # Carry forward the latest available price
    prices = prices.reindex(dates).ffill()

    cash = float(initial_capital)
    active_positions = []
    completed_trades = []
    daily_records = []

    # 4. Run simulation
    for date in dates:

        # Close positions whose exit date has arrived
        remaining_positions = []

        for position in active_positions:

            if position["Exit_Date"] <= date:

                exit_price = prices.at[
                    date, position["Ticker"]
                ]

                if pd.isna(exit_price):
                    raise ValueError(
                        f"Missing exit price: "
                        f"{position['Ticker']} on {date}"
                    )

                gross_profit = (
                    position["Investment"]
                    * (exit_price / position["Entry_Price"] - 1)
                )

                fees = position["Investment"] * round_trip_fee

                net_profit = gross_profit - fees

                cash += position["Investment"] + net_profit

                position["Exit_Price"] = exit_price
                position["Gross_Profit"] = gross_profit
                position["Fees"] = fees
                position["Net_Profit"] = net_profit
                position["Cash_After_Exit"] = cash

                completed_trades.append(position)

            else:
                remaining_positions.append(position)

        active_positions = remaining_positions

        # Open new positions
        todays_trades = trades[
            trades["Entry_Date"] == date
        ]

        for _, trade in todays_trades.iterrows():

            if len(active_positions) >= max_open_positions:
                break

            ticker = trade["Ticker"]

            entry_price = prices.at[date, ticker]

            if pd.isna(entry_price) or entry_price <= 0:
                raise ValueError(
                    f"Missing entry price: {ticker} on {date}"
                )

            investment = cash * position_fraction

            if investment <= 0 or cash < investment:
                continue

            position = {
                "Ticker": ticker,
                "Entry_Date": date,
                "Exit_Date": trade["Exit_Date"],
                "Investment": investment,
                "Entry_Price": entry_price
            }

            cash -= investment
            active_positions.append(position)

        # Calculate daily market value of open positions
        invested_capital = sum(
            p["Investment"] for p in active_positions
        )

        market_value = 0.0

        for position in active_positions:

            current_price = prices.at[
                date, position["Ticker"]
            ]

            if pd.isna(current_price):
                raise ValueError(
                    f"Missing market price: "
                    f"{position['Ticker']} on {date}"
                )

            position_value = (
                position["Investment"]
                * current_price / position["Entry_Price"]
            )

            market_value += position_value

        equity = cash + market_value

        daily_records.append({
            "Date": date,
            "Cash": cash,
            "Invested_Capital": invested_capital,
            "Market_Value": market_value,
            "Equity": equity,
            "Open_Positions": len(active_positions)
        })

    # 5. Create results
    results = pd.DataFrame(completed_trades)

    portfolio = pd.DataFrame(daily_records)

    portfolio = portfolio.set_index("Date").sort_index()

    # 6. Summary
    if not results.empty:
        total_profit = results["Net_Profit"].sum()
        win_rate = (
            results["Net_Profit"] > 0
        ).mean() * 100
    else:
        total_profit = 0
        win_rate = 0

    final_equity = (
        portfolio["Equity"].iloc[-1]
        if not portfolio.empty else initial_capital
    )
    
    if trading_dates is not None:
        dates = sorted(set(pd.to_datetime(trading_dates)))
    else:
        dates = sorted(
        set(trades["Entry_Date"]).union(trades["Exit_Date"]))
    summary = {
        "Initial Capital": initial_capital,
        "Final Cash": cash,
        "Total Trades": len(results),
        "Net Profit": total_profit,
        "Final Equity": final_equity,
        "Total Return (%)": (
            final_equity / initial_capital - 1
        ) * 100,
        "Win Rate (%)": win_rate,
        "Max Open Positions": (
            portfolio["Open_Positions"].max()
            if not portfolio.empty else 0
        ),
        "Trading Dates" : trading_dates,
        "Remaining Open Positions": len(active_positions),
        "First Date": portfolio.index.min() if not portfolio.empty else None,
        "Last Date": portfolio.index.max() if not portfolio.empty else None
    }

    return results, portfolio, pd.Series(summary)

In [270]:
start_date = strategy1_trades["Entry_Date"].min()
end_date = strategy1_trades["Exit_Date"].max()

trading_dates = sorted(
    daily_data.loc[
        daily_data["Date"].between(start_date, end_date),
        "Date"
    ].unique()
)

In [271]:
exact_trades, portfolio_s1, exact_summary = exact_simulation(
    strategy1_trades,
    daily_data=daily_data,
    initial_capital=1500,
    max_open_positions=15,
    position_fraction=1 / 15,
    trading_dates=trading_dates
)

display(exact_summary.to_frame("Value"))
display(portfolio_s1.head())
display(portfolio_s1.tail())
display(exact_summary)

,Value
Initial Capital,1500
Final Cash,1380.772725
Total Trades,4044
Net Profit,-119.227275
Final Equity,1380.772725
Total Return (%),-7.948485
Win Rate (%),54.129575
Max Open Positions,15
Trading Dates,"[2021-06-17 00:00:00, 2021-06-18 00:00:00, 202..."
Remaining Open Positions,0


,Cash,Invested_Capital,Market_Value,Equity,Open_Positions
Date,,,,,
2021-06-17,1219.555556,280.444444,280.444444,1500.000000,3
2021-06-18,991.543835,508.456165,506.072480,1497.616315,6
2021-06-21,806.161862,693.838138,694.998957,1501.160820,9
2021-06-22,655.439452,844.560548,846.871986,1502.311437,12
2021-06-23,532.896550,967.103450,967.615758,1500.512308,15


,Cash,Invested_Capital,Market_Value,Equity,Open_Positions
Date,,,,,
2026-11-02,804.866046,606.445602,591.047306,1395.913352,12
2026-11-03,945.452241,454.663881,441.199627,1386.651867,9
2026-11-04,1089.098396,302.918986,295.836073,1384.934469,6
2026-11-05,1232.180591,151.173639,148.913485,1381.094076,3
2026-11-06,1380.772725,0.000000,0.000000,1380.772725,0


Initial Capital                                                          1500
Final Cash                                                        1380.772725
Total Trades                                                             4044
Net Profit                                                        -119.227275
Final Equity                                                      1380.772725
Total Return (%)                                                    -7.948485
Win Rate (%)                                                        54.129575
Max Open Positions                                                         15
Trading Dates               [2021-06-17 00:00:00, 2021-06-18 00:00:00, 202...
Remaining Open Positions                                                    0
First Date                                                2021-06-17 00:00:00
Last Date                                                 2026-11-06 00:00:00
dtype: object

In [273]:
print(strategy1_trades.columns.tolist())

['Date', 'Ticker', 'Future_Outperformance_5D', 'Stock_Outperformance_Prob', 'Future_Return_5D', 'Close_Momentum_5D', 'Exit_Date', 'Investment', 'Gross_Profit', 'Fees', 'Net_Profit', 'Entry_Date']


In [274]:
print(daily_data[
    ["Date", "Ticker", "Adj Close"]
].head(10))

        Date  Ticker  Adj Close
0 2000-01-04  ABX.TO  17.804482
1 2000-01-05  ABX.TO  17.944401
2 2000-01-06  ABX.TO  18.049345
3 2000-01-07  ABX.TO  17.664574
4 2000-01-10  ABX.TO  17.664574
5 2000-01-11  ABX.TO  17.909424
6 2000-01-12  ABX.TO  17.699556
7 2000-01-13  ABX.TO  17.489670
8 2000-01-14  ABX.TO  17.244814
9 2000-01-17  ABX.TO  17.454699


# Sharpe Ratio , Maximum Drawdown , Rolling Returns

In [256]:
start_date = strategy1_trades["Entry_Date"].min()
end_date = strategy1_trades["Exit_Date"].max()

trading_dates = sorted(
    daily_data.loc[
        daily_data["Date"].between(start_date, end_date),
        "Date"
    ].unique()
)

exact_trades, portfolio_s1, exact_summary = exact_simulation(
    strategy1_trades,
    daily_data=daily_data,
    initial_capital=1500,
    max_open_positions=15,
    position_fraction=1 / 15,
    trading_dates=trading_dates
)

In [250]:
def calculate_risk_metrics(
    portfolio,
    initial_capital=1500,
    risk_free_rate=0.0
):
    df = portfolio.copy()

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    # Equity curve
    df["Equity"] = df["Equity"].astype(float)

    # Daily returns
    df["Daily_Return"] = df["Equity"].pct_change()

    # Sharpe Ratio
    daily_rf = risk_free_rate / 252

    excess_returns = df["Daily_Return"].dropna() - daily_rf

    if excess_returns.std() != 0 and len(excess_returns) > 1:
        sharpe = (
            excess_returns.mean()
            / excess_returns.std()
            * np.sqrt(252)
        )
    else:
        sharpe = np.nan

    # Maximum Drawdown
    df["Running_Peak"] = df["Equity"].cummax()

    df["Drawdown"] = (
        df["Equity"] / df["Running_Peak"] - 1
    )

    max_drawdown = df["Drawdown"].min() * 100

    # Rolling Returns
    rolling_periods = {
        "63D": 63,
        "126D": 126,
        "252D": 252
    }

    for name, period in rolling_periods.items():
        df[f"Rolling_Return_{name}"] = (
            df["Equity"] / df["Equity"].shift(period) - 1
        ) * 100

    metrics = {
        "Sharpe Ratio": sharpe,
        "Maximum Drawdown (%)": max_drawdown,
        "Best 63D Rolling Return (%)": df["Rolling_Return_63D"].max(),
        "Worst 63D Rolling Return (%)": df["Rolling_Return_63D"].min(),
        "Best 126D Rolling Return (%)": df["Rolling_Return_126D"].max(),
        "Worst 126D Rolling Return (%)": df["Rolling_Return_126D"].min(),
        "Best 252D Rolling Return (%)": df["Rolling_Return_252D"].max(),
        "Worst 252D Rolling Return (%)": df["Rolling_Return_252D"].min()
    }

    return df, pd.Series(metrics)

In [275]:
portfolio_s1 = portfolio_s1.reset_index()
portfolio_risk_s1, risk_metrics = calculate_risk_metrics(
    portfolio_s1,
    initial_capital=1500
)

display(risk_metrics)

Sharpe Ratio                     -0.104231
Maximum Drawdown (%)            -21.682670
Best 63D Rolling Return (%)      10.247156
Worst 63D Rolling Return (%)    -13.957595
Best 126D Rolling Return (%)     13.116230
Worst 126D Rolling Return (%)   -16.378781
Best 252D Rolling Return (%)     12.335004
Worst 252D Rolling Return (%)   -14.804335
dtype: float64

In [277]:
print("Trades:", len(strategy1_trades))

print("Entry Date:")
print(strategy1_trades["Entry_Date"].agg(["min", "max"]))

print("\nExit Date:")
strategy1_trades["Exit_Date"].agg(["min", "max"])

print("\nDaily Data:")
print(daily_data["Date"].agg(["min", "max"]))

print("\nTrading Dates:", len(trading_dates))

Trades: 4044
Entry Date:
min   2021-06-17
max   2026-10-30
Name: Entry_Date, dtype: datetime64[us]

Exit Date:

Daily Data:
min   2000-01-04
max   2026-11-06
Name: Date, dtype: datetime64[us]

Trading Dates: 1353


In [278]:
print(portfolio_s1[["Date", "Equity"]].to_string(index=False))

      Date      Equity
2021-06-17 1500.000000
2021-06-18 1497.616315
2021-06-21 1501.160820
2021-06-22 1502.311437
2021-06-23 1500.512308
2021-06-24 1512.435946
2021-06-25 1516.634963
2021-06-28 1506.269471
2021-06-29 1509.415536
2021-06-30 1507.557366
2021-07-02 1509.908152
2021-07-05 1512.117195
2021-07-06 1509.285990
2021-07-07 1512.980296
2021-07-08 1506.984558
2021-07-09 1513.254425
2021-07-12 1508.883466
2021-07-13 1510.825124
2021-07-14 1509.694347
2021-07-15 1512.498190
2021-07-16 1502.841222
2021-07-19 1486.521200
2021-07-20 1491.372589
2021-07-21 1499.001358
2021-07-22 1497.639792
2021-07-23 1501.590033
2021-07-26 1502.631361
2021-07-27 1506.744825
2021-07-28 1506.672196
2021-07-29 1509.880496
2021-07-30 1508.320469
2021-08-03 1510.163407
2021-08-04 1507.625359
2021-08-05 1503.430578
2021-08-06 1502.169060
2021-08-09 1500.197631
2021-08-10 1498.107299
2021-08-11 1503.880420
2021-08-12 1504.071387
2021-08-13 1501.588216
2021-08-16 1500.299474
2021-08-17 1497.063965
2021-08-18 

# Risk Management – Stop Loss

In [80]:
def apply_stop_loss(
    trades_df,
    daily_data,
    stop_loss_pct=0.03
):
    trades = trades_df.copy()
    prices = daily_data.copy()

    trades["Entry_Date"] = pd.to_datetime(trades["Entry_Date"])
    trades["Exit_Date"] = pd.to_datetime(trades["Exit_Date"])

    prices["Date"] = pd.to_datetime(prices["Date"])

    prices = prices[
        ["Date", "Ticker", "Adj Close"]
    ].dropna()

    prices = prices.sort_values(["Ticker", "Date"])

    price_groups = {
        ticker: group.set_index("Date")["Adj Close"].sort_index()
        for ticker, group in prices.groupby("Ticker")
    }

    actual_exit_dates = []
    actual_returns = []
    stop_loss_triggered = []

    for _, trade in trades.iterrows():

        ticker = trade["Ticker"]
        entry_date = trade["Entry_Date"]
        scheduled_exit = trade["Exit_Date"]

        ticker_prices = price_groups.get(ticker)

        if ticker_prices is None:
            raise ValueError(f"Missing price data for {ticker}")

        if entry_date not in ticker_prices.index:
            raise ValueError(
                f"Missing entry price for {ticker} on {entry_date}"
            )

        entry_price = ticker_prices.loc[entry_date]

        future_prices = ticker_prices.loc[
            (ticker_prices.index > entry_date)
            & (ticker_prices.index <= scheduled_exit)
        ]

        stop_price = entry_price * (1 - stop_loss_pct)

        hit = future_prices[future_prices <= stop_price]

        if not hit.empty:

            exit_date = hit.index[0]
            exit_price = hit.iloc[0]
            stopped = True

        else:

            if scheduled_exit not in ticker_prices.index:
                raise ValueError(
                    f"Missing scheduled exit price for {ticker} "
                    f"on {scheduled_exit}"
                )

            exit_date = scheduled_exit
            exit_price = ticker_prices.loc[scheduled_exit]
            stopped = False

        realized_return = (
            (exit_price / entry_price) - 1
        ) * 100

        actual_exit_dates.append(exit_date)
        actual_returns.append(realized_return)
        stop_loss_triggered.append(stopped)

    trades["Exit_Date"] = actual_exit_dates
    trades["Realized_Return_Pct"] = actual_returns
    trades["Stop_Loss_Triggered"] = stop_loss_triggered

    # Use the realized return instead of the original 5-day return
    trades["Future_Return_5D"] = trades["Realized_Return_Pct"]

    return trades

In [280]:
stop_loss_trades = apply_stop_loss(
    strategy1_trades,
    daily_data,
    stop_loss_pct=0.03
)

print(
    "Stop Loss Triggered:",
    stop_loss_trades["Stop_Loss_Triggered"].sum()
)

print(
    "Stop Loss Percentage:",
    stop_loss_trades["Stop_Loss_Triggered"].mean() * 100
)

Stop Loss Triggered: 832
Stop Loss Percentage: 20.573689416419384


In [281]:
stop_loss_results, stop_loss_portfolio_s1, stop_loss_summary = (
    exact_simulation(
        stop_loss_trades,
        daily_data=daily_data,
        initial_capital=1500,
        max_open_positions=15,
        position_fraction=1 / 15,
        trading_dates=trading_dates
    )
)
stop_loss_portfolio_s1 = stop_loss_portfolio_s1.reset_index()
stop_loss_portfolio_risk_s1, stop_loss_risk = calculate_risk_metrics(
    stop_loss_portfolio_s1
)

display(stop_loss_summary)
display(stop_loss_risk)

Initial Capital                                                          1500
Final Cash                                                        1359.339814
Total Trades                                                             4044
Net Profit                                                        -140.660186
Final Equity                                                      1359.339814
Total Return (%)                                                    -9.377346
Win Rate (%)                                                        52.324431
Max Open Positions                                                         15
Trading Dates               [2021-06-17 00:00:00, 2021-06-18 00:00:00, 202...
Remaining Open Positions                                                    0
First Date                                                2021-06-17 00:00:00
Last Date                                                 2026-11-06 00:00:00
dtype: object

Sharpe Ratio                     -0.162170
Maximum Drawdown (%)            -20.607741
Best 63D Rolling Return (%)      10.424066
Worst 63D Rolling Return (%)    -11.134196
Best 126D Rolling Return (%)     12.351810
Worst 126D Rolling Return (%)   -14.076017
Best 252D Rolling Return (%)     15.369729
Worst 252D Rolling Return (%)   -13.527367
dtype: float64

In [282]:
comparison = pd.DataFrame({
    "Without Stop Loss": exact_summary,
    "With Stop Loss": stop_loss_summary
})

display(comparison)

,Without Stop Loss,With Stop Loss
Initial Capital,1500,1500
Final Cash,1380.772725,1359.339814
Total Trades,4044,4044
Net Profit,-119.227275,-140.660186
Final Equity,1380.772725,1359.339814
Total Return (%),-7.948485,-9.377346
Win Rate (%),54.129575,52.324431
Max Open Positions,15,15
Trading Dates,"[2021-06-17 00:00:00, 2021-06-18 00:00:00, 202...","[2021-06-17 00:00:00, 2021-06-18 00:00:00, 202..."
Remaining Open Positions,0,0


In [283]:
risk_comparison = pd.DataFrame({
    "Without Stop Loss": risk_metrics,
    "With Stop Loss": stop_loss_risk
})

display(risk_comparison)

,Without Stop Loss,With Stop Loss
Sharpe Ratio,-0.104231,-0.162170
Maximum Drawdown (%),-21.682670,-20.607741
Best 63D Rolling Return (%),10.247156,10.424066
Worst 63D Rolling Return (%),-13.957595,-11.134196
Best 126D Rolling Return (%),13.116230,12.351810
Worst 126D Rolling Return (%),-16.378781,-14.076017
Best 252D Rolling Return (%),12.335004,15.369729
Worst 252D Rolling Return (%),-14.804335,-13.527367


# Market Regime Prediction

### Market Regime Data Collection and Integration

In [38]:
import yfinance as yf
import pandas as pd
from pathlib import Path

TICKER = "^GSPTSE"

tsx = yf.download(
    TICKER,
    start="2000-01-01",
    auto_adjust=False,
    progress=False
)

# Handle possible MultiIndex columns
if isinstance(tsx.columns, pd.MultiIndex):
    tsx.columns = tsx.columns.get_level_values(0)

tsx = tsx.reset_index()
tsx.columns.name = None

tsx = tsx.rename(columns={
    "Date": "Date",
    "Open": "TSX_Open",
    "High": "TSX_High",
    "Low": "TSX_Low",
    "Close": "TSX_Close",
    "Adj Close": "TSX_Adj_Close",
    "Volume": "TSX_Volume"
})

tsx["Date"] = pd.to_datetime(tsx["Date"]).dt.tz_localize(None)

tsx = (
    tsx.sort_values("Date")
       .drop_duplicates(subset=["Date"])
       .reset_index(drop=True)
)

Path("data/raw").mkdir(parents=True, exist_ok=True)

tsx.to_csv(
    "data/raw/canada_tsx_composite.csv",
    index=False
)

print("Shape:", tsx.shape)
print("Date range:", tsx["Date"].min(), "to", tsx["Date"].max())
print("\nMissing values:")
print(tsx.isna().sum())

display(tsx.head())
display(tsx.tail())

Shape: (6752, 7)
Date range: 2000-01-04 00:00:00 to 2026-11-18 00:00:00

Missing values:
Date             0
TSX_Adj_Close    0
TSX_Close        0
TSX_High         0
TSX_Low          0
TSX_Open         0
TSX_Volume       0
dtype: int64


/home/fayyad/.venvs/dev/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/home/fayyad/.venvs/dev/lib/python3.13/site-packages/yfinance/multi.py:220: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns=ctx.isins, inplace=True)
/home/fayyad/.venvs/dev/lib/python3.13/site-packages/yfinance/multi.py:224: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.sort_index(level=0, ax

,Date,TSX_Adj_Close,TSX_Close,TSX_High,TSX_Low,TSX_Open,TSX_Volume
0,2000-01-04,8202.599609,8202.599609,8305.900391,8157.600098,8271.700195,59509800
1,2000-01-05,8119.399902,8119.399902,8147.200195,8018.600098,8100.200195,94545700
2,2000-01-06,8114.200195,8114.200195,8186.399902,8075.700195,8137.000000,79690100
3,2000-01-07,8429.400391,8429.400391,8430.000000,8193.400391,8193.400391,87693100
4,2000-01-10,8602.500000,8602.500000,8610.799805,8509.099609,8544.299805,83317000


,Date,TSX_Adj_Close,TSX_Close,TSX_High,TSX_Low,TSX_Open,TSX_Volume
6747,2026-11-12,36137.199219,36137.199219,36367.398438,36080.898438,36199.898438,231362700
6748,2026-11-13,36105.898438,36105.898438,36260.300781,35995.199219,36212.800781,252938600
6749,2026-11-16,35722.101562,35722.101562,36118.398438,35654.500000,36015.199219,382994800
6750,2026-11-17,35315.300781,35315.300781,35705.898438,35198.601562,35616.398438,311227100
6751,2026-11-18,34594.578125,34594.578125,35060.210938,34555.550781,35060.210938,178821295


In [41]:
tsx["Daily_Return"] = tsx["TSX_Close"].pct_change() * 100

print(tsx["Daily_Return"].describe())

print("\nDaily return distribution:")
print(tsx["Daily_Return"].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))

count    6751.000000
mean        0.027092
std         1.071559
min       -12.344690
25%        -0.425959
50%         0.075845
75%         0.543429
max        11.957070
Name: Daily_Return, dtype: float64

Daily return distribution:
0.01   -3.059663
0.05   -1.607200
0.25   -0.425959
0.50    0.075845
0.75    0.543429
0.95    1.444591
0.99    2.760251
Name: Daily_Return, dtype: float64


In [42]:
tsx = tsx.sort_values("Date").copy()

tsx["Future_Market_Return_5D"] = (
    tsx["TSX_Close"].shift(-5) / tsx["TSX_Close"] - 1
) * 100

In [43]:
conditions = [
    tsx["Future_Market_Return_5D"] > 1,
    tsx["Future_Market_Return_5D"] < -1
]

choices = ["Bullish", "Bearish"]

tsx["Market_Regime_5D"] = np.select(
    conditions,
    choices,
    default="Sideways"
)

tsx.loc[
    tsx["Future_Market_Return_5D"].isna(),
    "Market_Regime_5D"
] = np.nan

In [44]:
print(
    tsx["Market_Regime_5D"]
    .value_counts()
)

print(
    tsx["Market_Regime_5D"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Market_Regime_5D
Sideways    2996
Bullish     2196
Bearish     1555
Name: count, dtype: int64
Market_Regime_5D
Sideways    44.40
Bullish     32.55
Bearish     23.05
Name: proportion, dtype: float64


In [85]:
MARKET_FEATURES = ECONOMIC + CALENDAR

In [45]:
# Verify that economic values are consistent across stocks
consistency = (
    daily_data.groupby("Date")[MARKET_FEATURES]
    .nunique(dropna=False)
)

print("Maximum unique values per feature/date:")
print(consistency.max().max())

# Keep one row per date
economic_daily = (
    daily_data[
        ["Date"] + MARKET_FEATURES +
        ["Bank_Availability_Date", "Macro_Availability_Date"]
    ]
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Economic daily shape:", economic_daily.shape)

Maximum unique values per feature/date:
1
Economic daily shape: (6744, 34)


In [46]:
print(
    "Bank availability after Date:",
    (economic_daily["Bank_Availability_Date"] >
     economic_daily["Date"]).sum()
)

print(
    "Macro availability after Date:",
    (economic_daily["Macro_Availability_Date"] >
     economic_daily["Date"]).sum()
)

Bank availability after Date: 0
Macro availability after Date: 0


In [47]:
market_target = tsx[
    ["Date", "Future_Market_Return_5D", "Market_Regime_5D"]
].copy()

market_dataset = economic_daily.merge(
    market_target,
    on="Date",
    how="inner",
    validate="one_to_one"
)

market_dataset = market_dataset.sort_values("Date").reset_index(drop=True)

print("Market Dataset Shape:", market_dataset.shape)
print("Date Range:", market_dataset["Date"].min(),
      "to", market_dataset["Date"].max())

print("\nTarget Distribution:")
print(market_dataset["Market_Regime_5D"].value_counts())

print("\nMissing Features:")
print(
    market_dataset[MARKET_FEATURES]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Market Dataset Shape: (6744, 36)
Date Range: 2000-01-04 00:00:00 to 2026-11-06 00:00:00

Target Distribution:
Market_Regime_5D
Sideways    2996
Bullish     2196
Bearish     1552
Name: count, dtype: int64

Missing Features:
USD_CAD_Change_21D         4319
USD_CAD                    4285
Policy_Rate_Change_21D     2366
Yield_Spread               2364
Policy_Rate                2343
Bond_Yield_10Y              273
CPI                           0
GDP_YoY                       0
Employment                    0
Labour_Force                  0
Participation_Rate            0
Unemployment_Rate             0
Inflation_YoY                 0
GDP_Real_Millions_CAD         0
GDP_MoM                       0
BCPI_Energy                   0
BCPI_NonEnergy                0
BCPI_Total                    0
BCPI_Metals                   0
BCPI_Change_12M               0
BCPI_Forestry                 0
BCPI_Agriculture              0
BCPI_Fish                     0
Unemployment_Change_12M       0
BCPI_Meta

In [48]:
inconsistent_features = consistency.columns[
    consistency.max() > 1
].tolist()

print("Inconsistent features:", inconsistent_features)

for feature in inconsistent_features:
    print(f"\n{feature}")
    print(
        daily_data.groupby("Date")[feature]
        .nunique(dropna=False)
        .loc[lambda x: x > 1]
        .head(10)
    )

Inconsistent features: []


In [49]:
missing_features = [
    "USD_CAD",
    "USD_CAD_Change_21D",
    "Policy_Rate",
    "Policy_Rate_Change_21D",
    "Yield_Spread",
    "Bond_Yield_10Y"
]

print(
    economic_daily.assign(
        Year=economic_daily["Date"].dt.year
    )
    .groupby("Year")[missing_features]
    .apply(lambda x: x.isna().mean() * 100)
    .round(1)
)

      USD_CAD  USD_CAD_Change_21D  Policy_Rate  Policy_Rate_Change_21D  \
Year                                                                     
2000    100.0               100.0        100.0                   100.0   
2001    100.0               100.0        100.0                   100.0   
2002    100.0               100.0        100.0                   100.0   
2003    100.0               100.0        100.0                   100.0   
2004    100.0               100.0        100.0                   100.0   
2005    100.0               100.0        100.0                   100.0   
2006    100.0               100.0        100.0                   100.0   
2007    100.0               100.0        100.0                   100.0   
2008    100.0               100.0        100.0                   100.0   
2009    100.0               100.0         29.9                    38.2   
2010    100.0               100.0          0.0                     0.0   
2011    100.0               100.0     

In [50]:
print("Missing future returns:", tsx["Future_Market_Return_5D"].isna().sum())

print("Missing regime:", tsx["Market_Regime_5D"].isna().sum())

print("\nLast 7 observations:")
display(
    tsx[
        ["Date", "Future_Market_Return_5D", "Market_Regime_5D"]
    ].tail(7)
)

Missing future returns: 5
Missing regime: 5

Last 7 observations:


,Date,Future_Market_Return_5D,Market_Regime_5D
6745,2026-11-10,-2.364356,Bearish
6746,2026-11-11,-4.990487,Bearish
6747,2026-11-12,NaN,NaN
6748,2026-11-13,NaN,NaN
6749,2026-11-16,NaN,NaN
6750,2026-11-17,NaN,NaN
6751,2026-11-18,NaN,NaN


In [51]:
check_date = pd.Timestamp("2015-05-21")

print(
    daily_data.loc[
        daily_data["Date"] == check_date,
        ["Date", "Ticker", "Policy_Rate",
         "Policy_Rate_Change_21D"]
    ].sort_values("Ticker").to_string(index=False)
)

      Date  Ticker  Policy_Rate  Policy_Rate_Change_21D
2015-05-21  ABX.TO         0.75                     0.0
2015-05-21  BMO.TO         0.75                     0.0
2015-05-21   BN.TO         0.75                     0.0
2015-05-21  CNQ.TO         0.75                     0.0
2015-05-21  CNR.TO         0.75                     0.0
2015-05-21  ENB.TO         0.75                     0.0
2015-05-21   RY.TO         0.75                     0.0
2015-05-21 SHOP.TO         0.75                     0.0
2015-05-21   SU.TO         0.75                     0.0
2015-05-21   TD.TO         0.75                     0.0


In [52]:
consistency = (
    daily_data.groupby("Date")[MARKET_FEATURES]
    .nunique(dropna=False)
)

print("Maximum unique values per feature/date:",
      consistency.max().max())

print("Inconsistent features:",
      consistency.columns[consistency.max() > 1].tolist())

Maximum unique values per feature/date: 1
Inconsistent features: []


In [53]:
economic_daily = (
    daily_data[
        ["Date"] + MARKET_FEATURES +
        ["Bank_Availability_Date", "Macro_Availability_Date"]
    ]
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .reset_index(drop=True)
)

In [54]:
market_target = tsx[
    ["Date", "Future_Market_Return_5D", "Market_Regime_5D"]
].copy()

market_dataset = economic_daily.merge(
    market_target,
    on="Date",
    how="inner",
    validate="one_to_one"
)

In [55]:
print("Shape:", market_dataset.shape)
print("Date Range:",
      market_dataset["Date"].min(),
      "to",
      market_dataset["Date"].max())

print("\nTarget Distribution:")
print(market_dataset["Market_Regime_5D"].value_counts())

print("\nMissing Values:")
print(
    market_dataset[MARKET_FEATURES]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nDuplicate Dates:",
      market_dataset["Date"].duplicated().sum())

Shape: (6744, 36)
Date Range: 2000-01-04 00:00:00 to 2026-11-06 00:00:00

Target Distribution:
Market_Regime_5D
Sideways    2996
Bullish     2196
Bearish     1552
Name: count, dtype: int64

Missing Values:
USD_CAD_Change_21D        4319
USD_CAD                   4285
Policy_Rate_Change_21D    2366
Yield_Spread              2364
Policy_Rate               2343
Bond_Yield_10Y             273
CPI                          0
GDP_YoY                      0
Employment                   0
Labour_Force                 0
dtype: int64

Duplicate Dates: 0


In [56]:
market_dataset.to_csv(
    "data/processed/canada_market_regime_dataset.csv",
    index=False
)

### Market Regime Data Collection and Integration

Historical data for the S&P/TSX Composite Index (`^GSPTSE`) was collected using `yfinance`, starting from January 2000. This index was selected to represent the Canadian equity market, which is the primary focus of the project.

The five-trading-day future market return (`Future_Market_Return_5D`) was calculated using the index closing prices and converted into three market regimes: Bullish, Bearish, and Sideways, based on thresholds of +1% and -1%.

The daily economic features were extracted from the integrated dataset using `MARKET_FEATURES`. Duplicate dates were removed to ensure one economic observation per trading day, while bank and macroeconomic availability dates were retained.

Finally, the economic dataset was merged with the market target variables using an inner join on `Date`, with one-to-one validation to ensure unique dates.

The resulting `market_dataset` combines daily economic indicators with future market returns and regime labels, providing the input dataset for the Market Regime Prediction model.

In [83]:
market_dataset = pd.read_csv(
    "data/processed/canada_market_regime_dataset.csv",
    parse_dates=["Date"]
)


## Market Regime Classification

#### Model Evaluation

In [323]:
MARKET_FEATURES = ECONOMIC + CALENDAR

In [86]:
from sklearn.model_selection import train_test_split

TARGET = "Market_Regime_5D"

X = market_dataset[MARKET_FEATURES].copy()
y = market_dataset[TARGET].copy()

dates = market_dataset["Date"]

n = len(market_dataset)
train_end = int(n * 0.60)
val_end = int(n * 0.80)

# 5-day purge gap
X_market_train = X.iloc[:train_end - 5]
y_market_train = y.iloc[:train_end - 5]

X_market_val = X.iloc[train_end:val_end - 5]
y_market_val = y.iloc[train_end:val_end - 5]

X_market_test = X.iloc[val_end:]
y_market_test = y.iloc[val_end:]

print("Train:", X_market_train.shape)
print("Validation:", X_market_val.shape)
print("Test:", X_market_test.shape)

print("\nDate ranges:")
print("Train:", dates.iloc[0], "->", dates.iloc[train_end - 6])
print("Validation:", dates.iloc[train_end], "->", dates.iloc[val_end - 6])
print("Test:", dates.iloc[val_end], "->", dates.iloc[-1])

print("\nTraining distribution:")
print(y_market_train.value_counts(normalize=True).round(3))

print("\nValidation distribution:")
print(y_market_val.value_counts(normalize=True).round(3))

print("\nTest distribution:")
print(y_market_test.value_counts(normalize=True).round(3))

Train: (4041, 31)
Validation: (1344, 31)
Test: (1349, 31)

Date ranges:
Train: 2000-01-04 00:00:00 -> 2016-01-29 00:00:00
Validation: 2016-02-08 00:00:00 -> 2021-06-15 00:00:00
Test: 2021-06-23 00:00:00 -> 2026-11-06 00:00:00

Training distribution:
Market_Regime_5D
Sideways    0.397
Bullish     0.342
Bearish     0.261
Name: proportion, dtype: float64

Validation distribution:
Market_Regime_5D
Sideways    0.572
Bullish     0.274
Bearish     0.154
Name: proportion, dtype: float64

Test distribution:
Market_Regime_5D
Sideways    0.459
Bullish     0.328
Bearish     0.213
Name: proportion, dtype: float64


In [87]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(
    strategy="median",
    add_indicator=True,
    keep_empty_features=True
)

In [88]:
MARKET_MODEL_FEATURES = [
    col for col in MARKET_FEATURES
    if col not in [
        "USD_CAD",
        "USD_CAD_Change_21D"
    ]
]

X_train_model = X_market_train[MARKET_MODEL_FEATURES]
X_val_model = X_market_val[MARKET_MODEL_FEATURES]
X_test_model = X_market_test[MARKET_MODEL_FEATURES]

print("Number of features:", len(MARKET_MODEL_FEATURES))

Number of features: 29


In [100]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Dummy": DummyClassifier(
        strategy="most_frequent"
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=30,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=20,
        max_features=0.8,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )
}

pipelines = {}

for name, model in models.items():

    pipelines[name] = Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True
        )),
        ("model", model)
    ])

In [101]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

results = []

for name, pipeline in pipelines.items():

    pipeline.fit(X_train_model, y_train)

    predictions = pipeline.predict(X_val_model)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Balanced Accuracy": balanced_accuracy_score(
            y_val, predictions
        ),
        "Macro F1": f1_score(
            y_val, predictions, average="macro"
        )
    })

    print(f"\n{name}")
    print(classification_report(
        y_val,
        predictions,
        zero_division=0
    ))

results_df = pd.DataFrame(results).sort_values(
    "Macro F1",
    ascending=False
)

print(results_df.to_string(index=False))


Dummy
              precision    recall  f1-score   support

     Bearish       0.00      0.00      0.00       207
     Bullish       0.00      0.00      0.00       368
    Sideways       0.57      1.00      0.73       769

    accuracy                           0.57      1344
   macro avg       0.19      0.33      0.24      1344
weighted avg       0.33      0.57      0.42      1344


Decision Tree
              precision    recall  f1-score   support

     Bearish       0.24      0.26      0.25       207
     Bullish       0.33      0.29      0.31       368
    Sideways       0.63      0.66      0.64       769

    accuracy                           0.49      1344
   macro avg       0.40      0.40      0.40      1344
weighted avg       0.49      0.49      0.49      1344


Random Forest
              precision    recall  f1-score   support

     Bearish       0.10      0.06      0.07       207
     Bullish       0.40      0.35      0.37       368
    Sideways       0.62      0.72     

In [102]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True,
        keep_empty_features=True
    )),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ))
])

In [103]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_market_train)
y_val_encoded = label_encoder.transform(y_market_val)

xgb_pipeline.fit(
    X_train_model,
    y_train_encoded
)

xgb_predictions = xgb_pipeline.predict(X_val_model)

xgb_predictions_labels = label_encoder.inverse_transform(
    xgb_predictions.astype(int)
)

print(classification_report(
    y_val,
    xgb_predictions_labels,
    zero_division=0
))

              precision    recall  f1-score   support

     Bearish       0.03      0.00      0.01       207
     Bullish       0.35      0.43      0.39       368
    Sideways       0.63      0.71      0.67       769

    accuracy                           0.53      1344
   macro avg       0.34      0.38      0.36      1344
weighted avg       0.47      0.53      0.49      1344



In [97]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_encoded
)

xgb_pipeline.fit(
    X_train_model,
    y_train_encoded,
    model__sample_weight=sample_weights
)

xgb_predictions = xgb_pipeline.predict(X_val_model)

xgb_predictions_labels = label_encoder.inverse_transform(
    xgb_predictions.astype(int)
)

print(classification_report(
    y_val,
    xgb_predictions_labels,
    zero_division=0
))

              precision    recall  f1-score   support

     Bearish       0.11      0.04      0.06       207
     Bullish       0.35      0.46      0.40       368
    Sideways       0.63      0.64      0.63       769

    accuracy                           0.50      1344
   macro avg       0.36      0.38      0.36      1344
weighted avg       0.47      0.50      0.48      1344



### Market Regime Classification – Model Evaluation
The economic indicators in MARKET_FEATURES were used to predict the Market_Regime_5D target, which represents Bullish, Bearish, and Sideways market conditions over the next five trading days.
The dataset was prepared for chronological splitting into training (60%), validation (20%), and testing (20%) sets.
Four approaches were evaluated: Dummy Classifier, Decision Tree, Random Forest, and XGBoost.
Decision Tree was selected based on its superior performance in the two metrics that account for class-level balance:
    • Balanced Accuracy: 0.4016
    • Macro F1: 0.4010
Although the Dummy Classifier achieved higher overall accuracy (57.22%), it failed to identify the Bullish and Bearish classes. Decision Tree provided a more balanced classification performance.
However, the model still struggles to identify Bearish conditions, and its predictions should be considered supplementary signals rather than highly reliable market forecasts.


# Tuning

In [111]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


# Economic Only features
#X_train = X_market_train[MARKET_FEATURES]
#X_val = X_market_val[MARKET_FEATURES]

# Pipeline
economic_tree_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True
        )
    ),
    (
        "model",
        DecisionTreeClassifier(random_state=42)
    )
])

# Hyperparameter grid
param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 5, 7, 10, 12],
    "model__min_samples_split": [20, 50, 100],
    "model__min_samples_leaf": [20, 30, 50],
    "model__class_weight": ["balanced", None]
}

# Time series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

# Grid search
grid_search = GridSearchCV(
    estimator=economic_tree_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=tscv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

# Fit on training data only
grid_search.fit(X_train_model, y_market_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV Macro F1:")
print(round(grid_search.best_score_, 4))

Fitting 5 folds for each of 180 candidates, totalling 900 fits
Best Parameters:
{'model__class_weight': None, 'model__criterion': 'entropy', 'model__max_depth': 10, 'model__min_samples_leaf': 20, 'model__min_samples_split': 100}

Best CV Macro F1:
0.2849


In [112]:
best_market_model = grid_search.best_estimator_

# Predictions
market_val_pred = best_market_model.predict(X_val_model)

# Metrics
print("Accuracy:",
      round(accuracy_score(y_market_val, market_val_pred), 4))

print("Balanced Accuracy:",
      round(balanced_accuracy_score(y_market_val, market_val_pred), 4))

print("Macro F1:",
      round(f1_score(y_market_val, market_val_pred, average="macro"), 4))

print("\nClassification Report:")
print(
    classification_report(
        y_market_val,
        market_val_pred,
        zero_division=0
    )
)

Accuracy: 0.4182
Balanced Accuracy: 0.351
Macro F1: 0.3366

Classification Report:
              precision    recall  f1-score   support

     Bearish       0.10      0.08      0.09       207
     Bullish       0.32      0.50      0.39       368
    Sideways       0.62      0.47      0.53       769

    accuracy                           0.42      1344
   macro avg       0.34      0.35      0.34      1344
weighted avg       0.46      0.42      0.42      1344



In [113]:
tuning_results = pd.DataFrame([
    {
        "Model": "Dummy",
        "Accuracy": 0.5722,
        "Balanced Accuracy": 0.3333,
        "Macro F1": 0.2426
    },
    {
        "Model": "Decision Tree - Before Tuning",
        "Accuracy": 0.4948,
        "Balanced Accuracy": 0.4016,
        "Macro F1": 0.4010
    },
    {
        "Model": "Decision Tree - Tuned",
        "Accuracy": accuracy_score(y_market_val, market_val_pred),
        "Balanced Accuracy": balanced_accuracy_score(
            y_market_val, market_val_pred
        ),
        "Macro F1": f1_score(
            y_market_val, market_val_pred, average="macro"
        )
    }
])

tuning_results.round(4)

,Model,Accuracy,Balanced Accuracy,Macro F1
0,Dummy,0.5722,0.3333,0.2426
1,Decision Tree - Before Tuning,0.4948,0.4016,0.4010
2,Decision Tree - Tuned,0.4182,0.3510,0.3366


### Market Regime Model Evaluation

The target distribution was examined across the training, validation, and test datasets. Sideways was the most frequent class in all three datasets, representing 39.7%, 57.2%, and 45.9%, respectively. The differences in class proportions indicate variation in market regime distribution across chronological periods.

Three classification models were evaluated: Decision Tree, Random Forest, and XGBoost. A Dummy Classifier was also used as a baseline.

| Model | Accuracy | Balanced Accuracy | Macro F1 |
|---|---:|---:|---:|
| Decision Tree | 0.4948 | **0.4016** | **0.4010** |
| Random Forest | 0.5186 | 0.3772 | 0.3710 |
| Dummy | 0.5722 | 0.3333 | 0.2426 |

The Dummy Classifier achieved the highest Accuracy by predicting only the majority class, Sideways, while completely failing to identify Bearish and Bullish conditions.

Decision Tree achieved the highest Balanced Accuracy and Macro F1, demonstrating the best overall balance across the three market regimes. Random Forest achieved higher Accuracy, while XGBoost showed better recall for Bullish conditions.

**Decision Tree was selected as the final model for Strategy 2**, prioritizing balanced performance across all classes rather than overall Accuracy.

The selected Decision Tree model provides market regime predictions that will be integrated with the stock outperformance predictions from Strategy 1.

The combined approach aims to evaluate whether incorporating broader Canadian market conditions can improve stock selection and overall trading performance compared with using stock predictions alone.

The effectiveness of this integration will be assessed through the final trading simulation.

# Strategy 1

### Market Regime Classification

In [89]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# Preprocessing
market_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True
        )
    ),
    (
        "model",
        DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=30,
            class_weight="balanced",
            random_state=42
        )
    )
])

# Train only
market_model.fit(X_train_model, y_market_train)

# Predict probabilities
market_val_probabilities = market_model.predict_proba(X_val_model)
market_test_probabilities = market_model.predict_proba(X_test_model)

# Save validation predictions
market_val_predictions = market_dataset.iloc[
    train_end:val_end - 5
][["Date", TARGET]].copy()

for i, class_name in enumerate(
    market_model.named_steps["model"].classes_
):
    market_val_predictions[f"Prob_{class_name}"] = (
        market_val_probabilities[:, i]
    )


# Save test predictions
market_test_predictions = market_dataset.iloc[
    val_end:
][["Date", TARGET]].copy()

for i, class_name in enumerate(
    market_model.named_steps["model"].classes_
):
    market_test_predictions[f"Prob_{class_name}"] = (
        market_test_probabilities[:, i]
    )
print("Validation predictions:", market_val_predictions.shape)
print("Test predictions:", market_test_predictions.shape)

print(market_val_predictions.head())

Validation predictions: (1344, 5)
Test predictions: (1349, 5)
           Date Market_Regime_5D  Prob_Bearish  Prob_Bullish  Prob_Sideways
4046 2016-02-08         Sideways      0.362575      0.364848       0.272577
4047 2016-02-09          Bullish      0.362575      0.364848       0.272577
4048 2016-02-10          Bullish      0.362575      0.364848       0.272577
4049 2016-02-11          Bullish      0.362575      0.364848       0.272577
4050 2016-02-12          Bullish      0.362575      0.364848       0.272577


In [117]:
print("Validation:", market_val_predictions.shape)
print("Test:", market_test_predictions.shape)

display(market_val_predictions.head())
display(market_test_predictions.head())

Validation: (1344, 5)
Test: (1349, 5)


,Date,Market_Regime_5D,Prob_Bearish,Prob_Bullish,Prob_Sideways
4046,2016-02-08,Sideways,0.362575,0.364848,0.272577
4047,2016-02-09,Bullish,0.362575,0.364848,0.272577
4048,2016-02-10,Bullish,0.362575,0.364848,0.272577
4049,2016-02-11,Bullish,0.362575,0.364848,0.272577
4050,2016-02-12,Bullish,0.362575,0.364848,0.272577


,Date,Market_Regime_5D,Prob_Bearish,Prob_Bullish,Prob_Sideways
5395,2021-06-23,Sideways,0.226493,0.270796,0.502711
5396,2021-06-24,Sideways,0.226493,0.270796,0.502711
5397,2021-06-25,Sideways,0.226493,0.270796,0.502711
5398,2021-06-28,Sideways,0.226493,0.270796,0.502711
5399,2021-06-29,Sideways,0.226493,0.270796,0.502711


In [34]:
# Check probability diversity
prob_columns = [
    "Prob_Bearish",
    "Prob_Bullish",
    "Prob_Sideways"
]

print("Unique probability combinations:")
print(
    market_val_predictions[prob_columns]
    .drop_duplicates()
    .shape[0]
)

print("\nUnique predicted regimes:")
print(
    market_val_predictions[prob_columns]
    .idxmax(axis=1)
    .value_counts()
)

print("\nProbability summary:")
display(
    market_val_predictions[prob_columns].describe()
)

Unique probability combinations:
10

Unique predicted regimes:
Prob_Sideways    802
Prob_Bullish     323
Prob_Bearish     219
Name: count, dtype: int64

Probability summary:


,Prob_Bearish,Prob_Bullish,Prob_Sideways
count,1344.000000,1344.000000,1344.000000
mean,0.245865,0.296330,0.457805
std,0.123452,0.068316,0.163997
min,0.122043,0.170235,0.221405
25%,0.159079,0.270796,0.272577
50%,0.161214,0.302557,0.515449
75%,0.281218,0.331880,0.536229
max,0.517808,0.482779,0.707722


In [35]:
from sklearn.metrics import confusion_matrix

print(
    confusion_matrix(
        market_val_predictions[TARGET],
        market_val_predictions[
            ["Prob_Bearish", "Prob_Bullish", "Prob_Sideways"]
        ].idxmax(axis=1).str.replace("Prob_", ""),
        labels=["Bearish", "Bullish", "Sideways"]
    )
)

[[ 53  44 110]
 [ 72 108 188]
 [ 94 171 504]]


### Market Regime Classification

A Decision Tree Classifier was developed to predict the Canadian market regime over the next five trading days, classifying market conditions into **Bearish, Bullish, and Sideways**.

A preprocessing pipeline using `SimpleImputer` was applied to handle missing numerical values through median imputation, while also adding missing-value indicators and preserving empty features.

The model was configured with `max_depth=5`, `min_samples_leaf=30`, `class_weight="balanced"`, and `random_state=42`.

The pipeline was trained exclusively on the training dataset. Subsequently, `predict_proba()` was used to generate class probabilities for the validation and test datasets.

The resulting prediction datasets contain the trading date, actual market regime, and predicted probabilities for each of the three classes.

These probabilities provide a market-level signal that can be combined with stock outperformance predictions in Strategy 2 to evaluate the impact of market conditions on trading decisions.


In [201]:
def market_rigime_strategy(market_test_predictions, stock_test_predictions):

    # 1. Select market regime prediction columns
    market_prob_cols = [
        "Prob_Bearish",
        "Prob_Bullish",
        "Prob_Sideways"
    ]

    market_signal = market_test_predictions[
        ["Date"] + market_prob_cols
    ].copy()

    # 2. Determine the predicted market regime
    market_signal["Predicted_Market_Regime"] = (
        market_signal[market_prob_cols]
        .idxmax(axis=1)
        .str.replace("Prob_", "", regex=False)
    )

    # 3. Merge market predictions with stock predictions
    strategy2_data = stock_test_predictions.merge(
        market_signal,
        on="Date",
        how="inner",
        validate="many_to_one"
    )

    # 4. Sort the combined dataset
    strategy2_data = (
        strategy2_data
        .sort_values(["Date", "Ticker"])
        .reset_index(drop=True)
    )
    return strategy2_data

In [203]:
market_strategy = market_rigime_strategy(market_test_predictions, stock_test_predictions)

# 5. Inspect the result
display(market_strategy.head())
print("Combined dataset shape:", market_strategy.shape)
print("\nPredicted Market Regime Distribution:")
display(
    market_strategy["Predicted_Market_Regime"]
    .value_counts(normalize=True)
    .round(3)
)

,Date,Ticker,Future_Outperformance_5D,Stock_Outperformance_Prob,Prob_Bearish,Prob_Bullish,Prob_Sideways,Predicted_Market_Regime
0,2021-06-23,ABX.TO,0.0,0.407303,0.226493,0.270796,0.502711,Sideways
1,2021-06-23,BMO.TO,0.0,0.529304,0.226493,0.270796,0.502711,Sideways
2,2021-06-23,BN.TO,1.0,0.529304,0.226493,0.270796,0.502711,Sideways
3,2021-06-23,CNQ.TO,1.0,0.787726,0.226493,0.270796,0.502711,Sideways
4,2021-06-23,CNR.TO,1.0,0.529304,0.226493,0.270796,0.502711,Sideways


Combined dataset shape: (13440, 8)

Predicted Market Regime Distribution:


Predicted_Market_Regime
Sideways    0.562
Bearish     0.387
Bullish     0.051
Name: proportion, dtype: float64

# Strategy 2 – Market-Aware Stock Selection Strategy

### Strategy 2 – Market-Aware Stock Selection Strategy

This strategy combines stock outperformance predictions with Canadian market regime classification to make investment decisions that account for both individual stock opportunities and overall market conditions.

Stocks are ranked according to their predicted outperformance probabilities, while the predicted market regime determines the investment allocation. The strategy maintains full exposure during Bullish conditions, reduces investment during Sideways conditions, and avoids opening new positions during Bearish conditions.

Selected positions are held for five trading days, with transaction fees included in the profit calculations. The simulation evaluates trading performance, capital requirements, and risk-related metrics.

#### Benchmark Comparison

The strategy is compared against the S&P/TSX Composite Index (^GSPTSE) over the same testing period using equivalent initial capital. This benchmark provides a passive market investment reference to evaluate whether the proposed strategy can outperform the broader Canadian equity market.

The objective is to improve stock selection and manage investment exposure by incorporating market conditions into the trading decision process.

In [204]:
trade_features = daily_data[
    [
        "Date",
        "Ticker",
        "Future_Return_5D",
        "Close_Momentum_5D"
    ]
].copy()

market_strategy = market_strategy.merge(
    trade_features,
    on=["Date", "Ticker"],
    how="inner",
    validate="one_to_one"
)

In [92]:
INITIAL_INVESTMENT = 100
TOP_N = 3
HOLDING_DAYS = 5
ROUND_TRIP_FEE = 0.002

REGIME_ALLOCATION = {
    "Bullish": 1.0,
    "Sideways": 0.5,
    "Bearish": 0.0
}

In [296]:
def simulate_market_aware_strategy(
    data,
    investment_per_stock=100,
    top_n=3,
    holding_days=5,
    round_trip_fee=0.002,
    regime_allocation=None
):

    if regime_allocation is None:
        regime_allocation = {
            "Bullish": 1.0,
            "Sideways": 0.5,
            "Bearish": 0.0
        }

    required_columns = [
        "Date",
        "Ticker",
        "Stock_Outperformance_Prob",
        "Predicted_Market_Regime",
        "Future_Return_5D",
        "Close_Momentum_5D"
    ]

    missing = set(required_columns) - set(data.columns)

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    trading_dates = sorted(data["Date"].unique())

    exit_dates = {
        date: trading_dates[i + holding_days]
        for i, date in enumerate(trading_dates)
        if i + holding_days < len(trading_dates)
    }

    trades = []

    for date, daily_group in data.groupby("Date", sort=True):

        if date not in exit_dates:
            continue

        regime = daily_group["Predicted_Market_Regime"].iloc[0]

        allocation = regime_allocation.get(regime, 0)

        if allocation <= 0:
            continue

        selected = (
            daily_group
            .sort_values(
                ["Stock_Outperformance_Prob", "Close_Momentum_5D"],
                ascending=[False, False]
            )
            .head(top_n)
            .copy()
        )

        selected["Investment"] = (
            investment_per_stock * allocation
        )

        selected["Gross_Profit"] = (
            selected["Investment"]
            * selected["Future_Return_5D"] / 100
        )

        selected["Fees"] = (
            selected["Investment"] * round_trip_fee
        )

        selected["Net_Profit"] = (
            selected["Gross_Profit"] - selected["Fees"]
        )
        selected["Date"] = date
        selected["Entry_Date"] = date
        selected["Exit_Date"] = exit_dates[date]

        trades.append(selected)

    if not trades:
        return pd.DataFrame(), {}

    trades_df = pd.concat(trades, ignore_index=True)

    total_trades = len(trades_df)
    
    total_profit = trades_df["Net_Profit"].sum()

    win_rate = (
        trades_df["Net_Profit"].gt(0).mean() * 100
    )

    average_profit = trades_df["Net_Profit"].mean()

    total_invested = trades_df["Investment"].sum()

    summary = {
        "Total Trades": total_trades,
        "Total Investment Across Trades ($)": round(total_invested, 2),
        "Total Gross Profit ($)": round(
            trades_df["Gross_Profit"].sum(), 2
        ),
        "Total Fees ($)": round(trades_df["Fees"].sum(), 2),
        "Total Net Profit ($)": round(total_profit, 2),
        "Win Rate (%)": round(win_rate, 2),
        "Average Net Profit per Trade ($)": round(
            average_profit, 4
        )
    }

    return trades_df, summary

In [297]:
strategy2_trades, strategy2_results = simulate_market_aware_strategy(
    data=market_strategy,
    investment_per_stock=INITIAL_INVESTMENT,
    top_n=TOP_N,
    holding_days=HOLDING_DAYS,
    round_trip_fee=ROUND_TRIP_FEE,
    regime_allocation=REGIME_ALLOCATION
)

print("Strategy 2 – Combined Strategy Results")

for metric, value in strategy2_results.items():
    print(f"{metric}: {value}")

display(strategy2_trades.head())

Strategy 2 – Combined Strategy Results
Total Trades: 2472
Total Investment Across Trades ($): 133800.0
Total Gross Profit ($): 493.73
Total Fees ($): 267.6
Total Net Profit ($): 226.13
Win Rate (%): 54.94
Average Net Profit per Trade ($): 0.0915


,Date,Ticker,Future_Outperformance_5D,Stock_Outperformance_Prob,Prob_Bearish,Prob_Bullish,Prob_Sideways,Predicted_Market_Regime,Future_Return_5D,Close_Momentum_5D,Investment,Gross_Profit,Fees,Net_Profit,Entry_Date,Exit_Date
0,2021-06-23,CNQ.TO,1.0,0.787726,0.226493,0.270796,0.502711,Sideways,0.829034,-1.717686,50.0,0.414517,0.1,0.314517,2021-06-23,2021-06-30
1,2021-06-23,BN.TO,1.0,0.529304,0.226493,0.270796,0.502711,Sideways,2.330094,0.536845,50.0,1.165047,0.1,1.065047,2021-06-23,2021-06-30
2,2021-06-23,TD.TO,0.0,0.529304,0.226493,0.270796,0.502711,Sideways,-0.149406,-0.956280,50.0,-0.074703,0.1,-0.174703,2021-06-23,2021-06-30
3,2021-06-24,BMO.TO,0.0,0.529304,0.226493,0.270796,0.502711,Sideways,-0.524404,-0.054760,50.0,-0.262202,0.1,-0.362202,2021-06-24,2021-07-02
4,2021-06-24,CNR.TO,1.0,0.529304,0.226493,0.270796,0.502711,Sideways,0.773650,-0.328289,50.0,0.386825,0.1,0.286825,2021-06-24,2021-07-02


In [95]:
strategy2_trades_200, strategy2_results_200 = (
    simulate_market_aware_strategy(
        data=strategy2_data,
        investment_per_stock=200
    )
)

print(strategy2_results_200)

{'Total Trades': 2472, 'Total Investment Across Trades ($)': np.float64(267600.0), 'Total Gross Profit ($)': np.float64(987.46), 'Total Fees ($)': np.float64(535.2), 'Total Net Profit ($)': np.float64(452.26), 'Win Rate (%)': np.float64(54.94), 'Average Net Profit per Trade ($)': np.float64(0.183)}


In [206]:
strategy2_trades_200, strategy2_results_200 = (
    simulate_market_aware_strategy(
        data=market_strategy,
        investment_per_stock=200
    )
)

print(strategy2_results_200)

{'Total Trades': 2472, 'Total Investment Across Trades ($)': np.float64(267600.0), 'Total Gross Profit ($)': np.float64(987.46), 'Total Fees ($)': np.float64(535.2), 'Total Net Profit ($)': np.float64(452.26), 'Win Rate (%)': np.float64(54.94), 'Average Net Profit per Trade ($)': np.float64(0.183)}


In [215]:
def calculate_capital_requirements(trades):

    trades = trades.copy()

    # ------------------------------------------
    # 1. Prepare dates
    # ------------------------------------------

    trades["Entry_Date"] = pd.to_datetime(trades["Entry_Date"])
    trades["Exit_Date"] = pd.to_datetime(trades["Exit_Date"])

    if trades.empty:
        return {
            "Start Date": None,
            "End Date": None,
            "Period (Years)": 0,
            "Total Net Profit ($)": 0,
            "Total Return (%)": 0,
            "CAGR (%)": np.nan,
            "Maximum Open Positions": 0,
            "Required Capital ($)": 0
        }

    # ------------------------------------------
    # 2. Calculate active capital
    # ------------------------------------------

    events = pd.concat([
        trades[["Entry_Date", "Investment"]].rename(
            columns={
                "Entry_Date": "Date",
                "Investment": "Change"
            }
        ),

        trades[["Exit_Date", "Investment"]].rename(
            columns={
                "Exit_Date": "Date",
                "Investment": "Change"
            }
        ).assign(
            Change=lambda x: -x["Change"]
        )
    ])

    daily_events = (
        events.groupby("Date")["Change"]
        .sum()
        .sort_index()
    )

    active_capital = daily_events.cumsum()

    required_capital = active_capital.max()

    # ------------------------------------------
    # 3. Calculate active positions
    # ------------------------------------------

    position_events = pd.concat([
        trades[["Entry_Date"]].rename(
            columns={"Entry_Date": "Date"}
        ).assign(Change=1),

        trades[["Exit_Date"]].rename(
            columns={"Exit_Date": "Date"}
        ).assign(Change=-1)
    ])

    daily_positions = (
        position_events.groupby("Date")["Change"]
        .sum()
        .sort_index()
        .cumsum()
    )

    max_open_positions = daily_positions.max()

    # ------------------------------------------
    # 4. Testing period
    # ------------------------------------------

    start_date = trades["Entry_Date"].min()
    end_date = trades["Exit_Date"].max()

    days = (end_date - start_date).days
    years = days / 365.25

    # ------------------------------------------
    # 5. Performance metrics
    # ------------------------------------------

    total_net_profit = trades["Net_Profit"].sum()

    total_return = (
        (total_net_profit / required_capital) * 100
        if required_capital > 0 else np.nan
    )

    if years > 0 and total_return > -100:
        cagr = (
            (1 + total_return / 100) ** (1 / years) - 1
        ) * 100
    else:
        cagr = np.nan

    # ------------------------------------------
    # 6. Return results
    # ------------------------------------------

    return {
        "Start Date": start_date,
        "End Date": end_date,
        "Period (Years)": round(years, 2),
        "Total Net Profit ($)": round(total_net_profit, 2),
        "Total Return (%)": round(total_return, 2),
        "CAGR (%)": round(cagr, 2),
        "Maximum Open Positions": int(max_open_positions),
        "Required Capital ($)": round(required_capital, 2)
    }

In [216]:
capital_results = calculate_capital_requirements(
    strategy2_trades
)

print("Strategy 2 – Capital Requirements")

for metric, value in capital_results.items():
    print(f"{metric}: {value}")

Strategy 2 – Capital Requirements
Start Date: 2021-06-23 00:00:00
End Date: 2026-09-01 00:00:00
Period (Years): 5.19
Total Net Profit ($): 226.13
Total Return (%): 15.08
CAGR (%): 2.74
Maximum Open Positions: 15
Required Capital ($): 1500.0


In [217]:
strategy2_results.update(capital_results)

display(
    pd.DataFrame(
        strategy2_results.items(),
        columns=["Metric", "Value"]
    )
)

,Metric,Value
0,Total Trades,2472
1,Total Investment Across Trades ($),133800.0
2,Total Gross Profit ($),493.73
3,Total Fees ($),267.6
4,Total Net Profit ($),226.13
5,Win Rate (%),54.94
6,Average Net Profit per Trade ($),0.0915
7,Start Date,2021-06-23 00:00:00
8,End Date,2026-09-01 00:00:00
9,Period (Years),5.19


In [299]:
vectorized_trades, vectorized_summary = vectorized_simulation(
    strategy2_trades,
    initial_capital=1500
)

display(vectorized_summary)

Total Trades                       2472.000000
Total Investment                 133800.000000
Gross Profit                        493.728138
Fees                                267.600000
Net Profit                          226.128138
Win Rate (%)                         54.935275
Average Net Profit                    0.091476
Initial Capital                    1500.000000
Return on Initial Capital (%)        15.075209
dtype: float64

In [300]:
trading_dates = dates = sorted(
    daily_data.loc[
        daily_data["Date"].between(
            strategy2_trades["Entry_Date"].min(),
            strategy2_trades["Entry_Date"].max()
        ),
        "Date"
    ].unique()
)


exact_trades, portfolio_s2, exact_summary = exact_simulation(
    strategy2_trades,
    daily_data=daily_data,
    initial_capital=1500,
    max_open_positions=15,
    position_fraction=1 / 15,
    trading_dates=trading_dates
)

display(exact_summary.to_frame("Value"))
display(portfolio_s2.head())
display(portfolio_s2.tail())

display(exact_summary)

,Value
Initial Capital,1500
Final Cash,1667.089739
Total Trades,2472
Net Profit,167.089739
Final Equity,1667.089739
Total Return (%),11.139316
Win Rate (%),54.935275
Max Open Positions,15
Trading Dates,"[2021-06-23 00:00:00, 2021-06-24 00:00:00, 202..."
Remaining Open Positions,0


,Cash,Invested_Capital,Market_Value,Equity,Open_Positions
Date,,,,,
2021-06-23,1219.555556,280.444444,280.444444,1500.000000,3
2021-06-24,991.543835,508.456165,512.221678,1503.765514,6
2021-06-25,806.161862,693.838138,701.124430,1507.286292,9
2021-06-28,655.439452,844.560548,843.555901,1498.995352,12
2021-06-29,532.896550,967.103450,970.483929,1503.380479,15


,Cash,Invested_Capital,Market_Value,Equity,Open_Positions
Date,,,,,
2026-08-20,1666.929746,0.000000,0.000000,1666.929746,0
2026-08-21,1666.929746,0.000000,0.000000,1666.929746,0
2026-08-24,1666.929746,0.000000,0.000000,1666.929746,0
2026-08-25,1355.275622,311.654124,311.654124,1666.929746,3
2026-09-01,1667.089739,0.000000,0.000000,1667.089739,0


Initial Capital                                                          1500
Final Cash                                                        1667.089739
Total Trades                                                             2472
Net Profit                                                         167.089739
Final Equity                                                      1667.089739
Total Return (%)                                                    11.139316
Win Rate (%)                                                        54.935275
Max Open Positions                                                         15
Trading Dates               [2021-06-23 00:00:00, 2021-06-24 00:00:00, 202...
Remaining Open Positions                                                    0
First Date                                                2021-06-23 00:00:00
Last Date                                                 2026-09-01 00:00:00
dtype: object

In [301]:
portfolio_s2 = portfolio_s2.reset_index()
portfolio_risk_s2, risk_metrics = calculate_risk_metrics(
    portfolio_s2,
    initial_capital=1500
)

display(risk_metrics)


Sharpe Ratio                      0.318801
Maximum Drawdown (%)            -12.859153
Best 63D Rolling Return (%)      10.247156
Worst 63D Rolling Return (%)     -7.153000
Best 126D Rolling Return (%)      8.241880
Worst 126D Rolling Return (%)    -8.589913
Best 252D Rolling Return (%)     10.724565
Worst 252D Rolling Return (%)    -9.392850
dtype: float64

In [302]:
stop_loss_trades = apply_stop_loss(
    strategy2_trades,
    daily_data,
    stop_loss_pct=0.03
)

print(
    "Stop Loss Triggered:",
    stop_loss_trades["Stop_Loss_Triggered"].sum()
)

Stop Loss Triggered: 416


In [303]:
print(
    "Stop Loss Percentage:",
    stop_loss_trades["Stop_Loss_Triggered"].mean() * 100
)


Stop Loss Percentage: 16.828478964401295


In [304]:
stop_loss_results, stop_loss_portfolio, stop_loss_summary = (
    exact_simulation(
        stop_loss_trades,
        daily_data=daily_data,
        initial_capital=1500,
        max_open_positions=15,
        position_fraction=1 / 15,
        trading_dates=trading_dates
    )
)
stop_loss_portfolio = stop_loss_portfolio.reset_index()
stop_loss_portfolio, stop_loss_risk = calculate_risk_metrics(
    stop_loss_portfolio
)

display(stop_loss_summary)
display(stop_loss_risk)


Initial Capital                                                          1500
Final Cash                                                        1583.637538
Total Trades                                                             2472
Net Profit                                                          83.637538
Final Equity                                                      1583.637538
Total Return (%)                                                     5.575836
Win Rate (%)                                                        53.195793
Max Open Positions                                                         15
Trading Dates               [2021-06-23 00:00:00, 2021-06-24 00:00:00, 202...
Remaining Open Positions                                                    0
First Date                                                2021-06-23 00:00:00
Last Date                                                 2026-09-01 00:00:00
dtype: object

Sharpe Ratio                      0.196453
Maximum Drawdown (%)            -11.991670
Best 63D Rolling Return (%)       7.323257
Worst 63D Rolling Return (%)     -7.968178
Best 126D Rolling Return (%)      7.515117
Worst 126D Rolling Return (%)    -9.544312
Best 252D Rolling Return (%)     10.092725
Worst 252D Rolling Return (%)    -7.458060
dtype: float64

In [305]:
comparison = pd.DataFrame({
    "Without Stop Loss": exact_summary,
    "With Stop Loss": stop_loss_summary
})

display(comparison)

risk_comparison = pd.DataFrame({
    "Without Stop Loss": risk_metrics,
    "With Stop Loss": stop_loss_risk
})

display(risk_comparison)


,Without Stop Loss,With Stop Loss
Initial Capital,1500,1500
Final Cash,1667.089739,1583.637538
Total Trades,2472,2472
Net Profit,167.089739,83.637538
Final Equity,1667.089739,1583.637538
Total Return (%),11.139316,5.575836
Win Rate (%),54.935275,53.195793
Max Open Positions,15,15
Trading Dates,"[2021-06-23 00:00:00, 2021-06-24 00:00:00, 202...","[2021-06-23 00:00:00, 2021-06-24 00:00:00, 202..."
Remaining Open Positions,0,0


,Without Stop Loss,With Stop Loss
Sharpe Ratio,0.318801,0.196453
Maximum Drawdown (%),-12.859153,-11.991670
Best 63D Rolling Return (%),10.247156,7.323257
Worst 63D Rolling Return (%),-7.153000,-7.968178
Best 126D Rolling Return (%),8.241880,7.515117
Worst 126D Rolling Return (%),-8.589913,-9.544312
Best 252D Rolling Return (%),10.724565,10.092725
Worst 252D Rolling Return (%),-9.392850,-7.458060


# Benchmark

In [181]:
tsx_data = pd.read_csv(
    "data/raw/canada_tsx_composite.csv",
    parse_dates=["Date"]
)

In [182]:
benchmark = (
    tsx_data[["Date", "TSX_Adj_Close"]]
    .dropna(subset=["TSX_Adj_Close"])
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .reset_index(drop=True)
)
benchmark["Date"] = pd.to_datetime(benchmark["Date"])

print(benchmark.info())
display(benchmark.head())

<class 'pandas.DataFrame'>
RangeIndex: 6752 entries, 0 to 6751
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           6752 non-null   datetime64[us]
 1   TSX_Adj_Close  6752 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 105.6 KB
None


,Date,TSX_Adj_Close
0,2000-01-04,8202.599609
1,2000-01-05,8119.399902
2,2000-01-06,8114.200195
3,2000-01-07,8429.400391
4,2000-01-10,8602.500000


In [219]:
def simulate_tsx(tsx_data, strategy, initial_capital=1500):

    # ------------------------------------------
    # 1. Prepare TSX benchmark data
    # ------------------------------------------

    benchmark = (
        tsx_data[
            ["Date", "TSX_Adj_Close"]
        ]
        .copy()
    )

    benchmark["Date"] = pd.to_datetime(benchmark["Date"])

    benchmark = (
        benchmark
        .dropna(subset=["TSX_Adj_Close"])
        .drop_duplicates(subset="Date", keep="last")
        .sort_values("Date")
        .reset_index(drop=True)
    )

    # ------------------------------------------
    # 2. Determine strategy testing period
    # ------------------------------------------

    strategy = strategy.copy()

    strategy["Entry_Date"] = pd.to_datetime(strategy["Entry_Date"])
    strategy["Exit_Date"] = pd.to_datetime(strategy["Exit_Date"])

    start_date = strategy["Entry_Date"].min()
    end_date = strategy["Exit_Date"].max()

    # ------------------------------------------
    # 3. Select common benchmark period
    # ------------------------------------------

    benchmark_period = benchmark[
        (benchmark["Date"] >= start_date) &
        (benchmark["Date"] <= end_date)
    ].copy()

    if len(benchmark_period) < 2:
        raise ValueError(
            "Insufficient TSX data for the strategy testing period."
        )

    benchmark_start = benchmark_period.iloc[0]
    benchmark_end = benchmark_period.iloc[-1]

    # ------------------------------------------
    # 4. Calculate benchmark performance
    # ------------------------------------------

    start_price = benchmark_start["TSX_Adj_Close"]
    end_price = benchmark_end["TSX_Adj_Close"]

    benchmark_return = (
        (end_price / start_price) - 1
    ) * 100

    benchmark_profit = (
        initial_capital * benchmark_return / 100
    )

    benchmark_final_equity = (
        initial_capital + benchmark_profit
    )

    # ------------------------------------------
    # 5. Calculate CAGR
    # ------------------------------------------

    testing_days = (
        benchmark_end["Date"] - benchmark_start["Date"]
    ).days

    testing_years = testing_days / 365.25

    benchmark_cagr = (
        (
            benchmark_final_equity / initial_capital
        ) ** (1 / testing_years) - 1
    ) * 100 if testing_years > 0 else np.nan

    # ------------------------------------------
    # 6. Return results
    # ------------------------------------------

    return {
        "Benchmark": "S&P/TSX Composite",
        "Start_Date": benchmark_start["Date"],
        "End_Date": benchmark_end["Date"],
        "Initial_Capital": initial_capital,
        "Final_Equity": round(benchmark_final_equity, 2),
        "Net_Profit": round(benchmark_profit, 2),
        "Total_Return": round(benchmark_return, 2),
        "Testing_Period_Years": round(testing_years, 2),
        "CAGR": round(benchmark_cagr, 2)
    }

In [240]:
# Strategy 1
tsx_strategy1 = simulate_tsx(
    tsx_data,
    strategy1_trades
)

# Strategy 2
tsx_strategy2 = simulate_tsx(
    tsx_data,
    strategy2_trades
)

In [241]:
display(
    pd.DataFrame([
        tsx_strategy1,
        tsx_strategy2
    ])
)

,Benchmark,Start_Date,End_Date,Initial_Capital,Final_Equity,Net_Profit,Total_Return,Testing_Period_Years,CAGR
0,S&P/TSX Composite,2021-06-17,2021-06-30,1500,1501.61,1.61,0.11,0.04,3.06
1,S&P/TSX Composite,2021-06-23,2026-09-01,1500,2665.02,1165.02,77.67,5.19,11.71


### Strategy 2 – Benchmark Comparison

The Market-Aware Stock Selection Strategy was evaluated against the S&P/TSX Composite Index (^GSPTSE) over the same testing period, from June 23, 2021, to September 1, 2026.

Both investments were evaluated using an initial capital of \$1,500.

The strategy generated a total net profit of \$226.13, corresponding to a total return of 15.08% and a CAGR of 2.74%.

In comparison, the S&P/TSX Composite achieved a total return of 77.67%, generating a benchmark profit of \$1,165.02 and a CAGR of 11.71%.

The benchmark outperformed the proposed strategy by 62.59 percentage points in total return.

These results indicate that although incorporating market regime predictions improved the strategy's performance relative to the stock-only approach, it did not outperform the broader Canadian equity market during the evaluation period.

Therefore, the S&P/TSX Composite serves as an important benchmark, highlighting the limitations of the proposed stock selection and market regime prediction approach.

### Benchmark Comparison Summary

Strategy 2 achieved a total return of 15.08% and a CAGR of 2.74%, compared with 77.67% and 11.71%, respectively, for the S&P/TSX Composite over the same testing period.

Despite improving upon Strategy 1, the market-aware approach significantly underperformed the Canadian market benchmark. This suggests that the current predictive models and allocation rules require further improvement before the strategy can compete with a passive market investment.

In [286]:
portfolio_strategy1 = portfolio_s1.copy()
portfolio_strategy2 = portfolio_s2.copy()

In [308]:
portfolio_s1.index = pd.to_datetime(portfolio_s1.index)
portfolio_s2.index = pd.to_datetime(portfolio_s2.index)

common_dates = portfolio_s1.index.intersection(
    portfolio_s2.index
).sort_values()

print("Common Start Date:", common_dates.min())
print("Common End Date:", common_dates.max())
print("Common Trading Days:", len(common_dates))

Common Start Date: 1970-01-01 00:00:00
Common End Date: 1970-01-01 00:00:00.000001298
Common Trading Days: 1299


In [291]:
portfolio_s1_common = portfolio_s1.loc[common_dates].copy()
portfolio_s2_common = portfolio_s2.loc[common_dates].copy()

In [292]:
print(portfolio_s1.columns.tolist())
print(portfolio_s1.head(3))

['Date', 'Cash', 'Invested_Capital', 'Market_Value', 'Equity', 'Open_Positions']
                                    Date         Cash  Invested_Capital  \
1970-01-01 00:00:00.000000000 2021-06-17  1219.555556        280.444444   
1970-01-01 00:00:00.000000001 2021-06-18   991.543835        508.456165   
1970-01-01 00:00:00.000000002 2021-06-21   806.161862        693.838138   

                               Market_Value       Equity  Open_Positions  
1970-01-01 00:00:00.000000000    280.444444  1500.000000               3  
1970-01-01 00:00:00.000000001    506.072480  1497.616315               6  
1970-01-01 00:00:00.000000002    694.998957  1501.160820               9  


In [293]:
print("Trading Dates:", len(trading_dates))

print("Strategy 1 Portfolio:", len(portfolio_s1))
print("Strategy 2 Portfolio:", len(portfolio_s2))

print("Strategy 1 Date Range:")
print(portfolio_s1.index.min(), portfolio_s1.index.max())

print("Strategy 2 Date Range:")
print(portfolio_s2.index.min(), portfolio_s2.index.max())

Trading Dates: 1298
Strategy 1 Portfolio: 1353
Strategy 2 Portfolio: 1299
Strategy 1 Date Range:
1970-01-01 00:00:00 1970-01-01 00:00:00.000001352
Strategy 2 Date Range:
2021-06-23 00:00:00 2026-09-01 00:00:00


In [306]:
portfolio_s1["Date"] = pd.to_datetime(portfolio_s1["Date"])
portfolio_s2["Date"] = pd.to_datetime(portfolio_s2["Date"])

portfolio_s1 = portfolio_s1.sort_values("Date").reset_index(drop=True)
portfolio_s2 = portfolio_s2.sort_values("Date").reset_index(drop=True)

In [307]:
start_date = max(
    portfolio_s1["Date"].min(),
    portfolio_s2["Date"].min()
)

end_date = min(
    portfolio_s1["Date"].max(),
    portfolio_s2["Date"].max()
)

print("Common Start:", start_date)
print("Common End:", end_date)

Common Start: 2021-06-23 00:00:00
Common End: 2026-09-01 00:00:00


In [309]:
portfolio_s1_common = portfolio_s1[
    portfolio_s1["Date"].between(start_date, end_date)
].copy()

portfolio_s2_common = portfolio_s2[
    portfolio_s2["Date"].between(start_date, end_date)
].copy()

In [310]:
print("Strategy 1:")
print(portfolio_s1_common["Date"].min())
print(portfolio_s1_common["Date"].max())
print(len(portfolio_s1_common))

print("\nStrategy 2:")
print(portfolio_s2_common["Date"].min())
print(portfolio_s2_common["Date"].max())
print(len(portfolio_s2_common))

Strategy 1:
2021-06-23 00:00:00
2026-09-01 00:00:00
1303

Strategy 2:
2021-06-23 00:00:00
2026-09-01 00:00:00
1299


In [311]:
dates_s1 = set(portfolio_s1_common["Date"])
dates_s2 = set(portfolio_s2_common["Date"])

missing_s2 = sorted(dates_s1 - dates_s2)
missing_s1 = sorted(dates_s2 - dates_s1)

print("Dates missing in Strategy 2:", len(missing_s2))
print("Dates missing in Strategy 1:", len(missing_s1))

print("\nFirst missing dates in Strategy 2:")
print(missing_s2[:10])

Dates missing in Strategy 2: 4
Dates missing in Strategy 1: 0

First missing dates in Strategy 2:
[Timestamp('2026-08-26 00:00:00'), Timestamp('2026-08-27 00:00:00'), Timestamp('2026-08-28 00:00:00'), Timestamp('2026-08-31 00:00:00')]


In [312]:
# Unified trading calendar
trading_calendar = portfolio_s1_common["Date"].sort_values().unique()

# Set Date as index
s1 = portfolio_s1_common.set_index("Date").sort_index()
s2 = portfolio_s2_common.set_index("Date").sort_index()

# Align both portfolios to the same trading calendar
equity_comparison = pd.DataFrame(index=trading_calendar)

equity_comparison["Strategy_1_Equity"] = (
    s1["Equity"].reindex(trading_calendar).ffill()
)

equity_comparison["Strategy_2_Equity"] = (
    s2["Equity"].reindex(trading_calendar).ffill()
)

# Restore initial equity if any leading values are missing
equity_comparison = equity_comparison.fillna(1500)

equity_comparison.index = pd.to_datetime(equity_comparison.index)
equity_comparison.index.name = "Date"

display(equity_comparison.head())
display(equity_comparison.tail())

,Strategy_1_Equity,Strategy_2_Equity
Date,,
2021-06-23,1500.512308,1500.000000
2021-06-24,1512.435946,1503.765514
2021-06-25,1516.634963,1507.286292
2021-06-28,1506.269471,1498.995352
2021-06-29,1509.415536,1503.380479


,Strategy_1_Equity,Strategy_2_Equity
Date,,
2026-08-26,1395.702683,1666.929746
2026-08-27,1396.191076,1666.929746
2026-08-28,1396.879194,1666.929746
2026-08-31,1391.255315,1666.929746
2026-09-01,1387.403970,1667.089739


In [313]:
print("Number of Trading Days:", len(equity_comparison))

print("Strategy 1 Missing:", equity_comparison["Strategy_1_Equity"].isna().sum())
print("Strategy 2 Missing:", equity_comparison["Strategy_2_Equity"].isna().sum())

print("Start Date:", equity_comparison.index.min())
print("End Date:", equity_comparison.index.max())

Number of Trading Days: 1303
Strategy 1 Missing: 0
Strategy 2 Missing: 0
Start Date: 2021-06-23 00:00:00
End Date: 2026-09-01 00:00:00


In [314]:
INITIAL_CAPITAL = 1500
TRADING_DAYS = 252

# Daily returns based on portfolio equity
daily_returns = equity_comparison.pct_change().fillna(0)

# Performance metrics
def calculate_performance_metrics(equity, returns, initial_capital=1500):

    final_equity = equity.iloc[-1]

    total_return = (
        (final_equity / initial_capital) - 1
    ) * 100

    years = len(equity) / TRADING_DAYS

    cagr = (
        (final_equity / initial_capital) ** (1 / years) - 1
    ) * 100

    volatility = returns.std() * np.sqrt(TRADING_DAYS) * 100

    sharpe = (
        returns.mean() / returns.std() * np.sqrt(TRADING_DAYS)
        if returns.std() != 0 else np.nan
    )

    rolling_max = equity.cummax()

    drawdown = (
        equity / rolling_max - 1
    ) * 100

    max_drawdown = drawdown.min()

    return {
        "Initial Capital": initial_capital,
        "Final Equity": final_equity,
        "Net Profit": final_equity - initial_capital,
        "Total Return (%)": total_return,
        "CAGR (%)": cagr,
        "Annualized Volatility (%)": volatility,
        "Sharpe Ratio": sharpe,
        "Maximum Drawdown (%)": max_drawdown
    }



In [315]:

# Calculate metrics for both strategies
s1_metrics = calculate_performance_metrics(
    equity_comparison["Strategy_1_Equity"],
    daily_returns["Strategy_1_Equity"],
    INITIAL_CAPITAL
)

s2_metrics = calculate_performance_metrics(
    equity_comparison["Strategy_2_Equity"],
    daily_returns["Strategy_2_Equity"],
    INITIAL_CAPITAL
)

comparison = pd.DataFrame({
    "Strategy 1: Top 3 Stocks": s1_metrics,
    "Strategy 2: Market-Aware": s2_metrics
})

display(comparison.round(4))

,Strategy 1: Top 3 Stocks,Strategy 2: Market-Aware
Initial Capital,1500.0000,1500.0000
Final Equity,1387.4040,1667.0897
Net Profit,-112.5960,167.0897
Total Return (%),-7.5064,11.1393
CAGR (%),-1.4978,2.0636
Annualized Volatility (%),10.0661,7.2434
Sharpe Ratio,-0.1002,0.3182
Maximum Drawdown (%),-21.6827,-12.8592


In [316]:
rolling_comparison = pd.DataFrame(index=equity_comparison.index)

for strategy in equity_comparison.columns:

    for window in [63, 126, 252]:

        rolling_return = (
            equity_comparison[strategy]
            / equity_comparison[strategy].shift(window)
            - 1
        ) * 100

        rolling_comparison[
            f"{strategy}_{window}D"
        ] = rolling_return

rolling_summary = pd.DataFrame({
    "Best Return": rolling_comparison.max(),
    "Worst Return": rolling_comparison.min()
})

display(rolling_summary.round(4))

,Best Return,Worst Return
Strategy_1_Equity_63D,10.2472,-13.9576
Strategy_1_Equity_126D,13.1162,-16.3788
Strategy_1_Equity_252D,12.3350,-14.8043
Strategy_2_Equity_63D,10.2472,-7.1530
Strategy_2_Equity_126D,8.2419,-8.5899
Strategy_2_Equity_252D,10.7246,-9.3929


In [327]:
def add_tsx_benchmark(
    tsx_data,
    equity_comparison,
    initial_capital=1500
):

    equity_comparison = equity_comparison.copy()

    # 1. Prepare TSX Data
    tsx = tsx_data.copy()

    tsx["Date"] = pd.to_datetime(tsx["Date"])

    tsx = (
        tsx.sort_values("Date")
        .drop_duplicates(subset="Date")
        .set_index("Date")
    )

    # 2. Determine Common Period
    equity_comparison.index = pd.to_datetime(
        equity_comparison.index
    )

    start_date = equity_comparison.index.min()
    end_date = equity_comparison.index.max()

    # 3. Align TSX with Portfolio Trading Dates
    tsx_common = tsx.loc[
        start_date:end_date
    ].copy()

    tsx_common = tsx_common.reindex(
        equity_comparison.index
    )

    tsx_common["TSX_Adj_Close"] = (
        tsx_common["TSX_Adj_Close"].ffill()
    )

    # 4. Calculate Benchmark Equity
    tsx_initial_price = tsx_common["TSX_Adj_Close"].iloc[0]
    tsx_final_price = tsx_common["TSX_Adj_Close"].iloc[-1]

    if pd.isna(tsx_initial_price) or tsx_initial_price <= 0:
        raise ValueError("Invalid TSX initial price.")

    tsx_final_equity = (
        initial_capital
        * tsx_final_price
        / tsx_initial_price
    )

    tsx_total_return = (
        (tsx_final_equity / initial_capital) - 1
    ) * 100

    years = (
        (end_date - start_date).days / 365.25
    )

    tsx_cagr = (
        (tsx_final_equity / initial_capital) ** (1 / years) - 1
    ) * 100 if years > 0 else np.nan

    # 5. Add Benchmark to Equity Comparison
    equity_comparison["TSX_Benchmark"] = (
        initial_capital
        * tsx_common["TSX_Adj_Close"]
        / tsx_initial_price
    )

    # 6. Display Results
    print("TSX Benchmark Results")
    print("--------------------")
    print(f"Initial Capital: ${initial_capital:.2f}")
    print(f"Final Equity: ${tsx_final_equity:.2f}")
    print(f"Net Profit: ${tsx_final_equity - initial_capital:.2f}")
    print(f"Total Return: {tsx_total_return:.2f}%")
    print(f"CAGR: {tsx_cagr:.2f}%")

    return equity_comparison

In [328]:
equity_comparison = add_tsx_benchmark(
    tsx_data=tsx_data,
    equity_comparison=equity_comparison,
    initial_capital=1500
)

TSX Benchmark Results
--------------------
Initial Capital: $1500.00
Final Equity: $2665.02
Net Profit: $1165.02
Total Return: 77.67%
CAGR: 11.71%


In [317]:
print(tsx_data.columns.tolist())
print(tsx_data.head())

['Date', 'TSX_Adj_Close', 'TSX_Close', 'TSX_High', 'TSX_Low', 'TSX_Open', 'TSX_Volume']
        Date  TSX_Adj_Close    TSX_Close     TSX_High      TSX_Low  \
0 2000-01-04    8202.599609  8202.599609  8305.900391  8157.600098   
1 2000-01-05    8119.399902  8119.399902  8147.200195  8018.600098   
2 2000-01-06    8114.200195  8114.200195  8186.399902  8075.700195   
3 2000-01-07    8429.400391  8429.400391  8430.000000  8193.400391   
4 2000-01-10    8602.500000  8602.500000  8610.799805  8509.099609   

      TSX_Open  TSX_Volume  
0  8271.700195    59509800  
1  8100.200195    94545700  
2  8137.000000    79690100  
3  8193.400391    87693100  
4  8544.299805    83317000  


In [326]:
benchmark_comparison = pd.DataFrame({
    "Strategy 1: Top 3": [
        equity_comparison["Strategy_1_Equity"].iloc[-1]
    ],
    "Strategy 2: Market-Aware": [
        equity_comparison["Strategy_2_Equity"].iloc[-1]
    ],
    "TSX Benchmark": [
        equity_comparison["TSX_Benchmark"].iloc[-1]
    ]
}, index=["Final Equity"]).T

benchmark_comparison["Net Profit"] = (
    benchmark_comparison["Final Equity"] - initial_capital
)

benchmark_comparison["Total Return (%)"] = (
    benchmark_comparison["Net Profit"] / initial_capital * 100
)

display(benchmark_comparison.round(4))

,Final Equity,Net Profit,Total Return (%)
Strategy 1: Top 3,1387.4040,-112.5960,-7.5064
Strategy 2: Market-Aware,1667.0897,167.0897,11.1393
TSX Benchmark,2665.0209,1165.0209,77.6681


# Trading Simulation & Strategy Evaluation

## 1. Simulation Objective

The objective of this stage was to evaluate the practical performance of the developed machine learning models by translating their predictions into executable trading strategies.

Two strategies were implemented and evaluated against the S&P/TSX Composite Index to determine whether incorporating market conditions improves stock selection and portfolio performance.

## 2. Trading Strategies

### Strategy 1: Top 3 Stock Selection

The baseline strategy relies exclusively on the Stock Outperformance Model.

At each trading date:
- Stocks are ranked according to their predicted outperformance probabilities.
- The three highest-ranked stocks are selected.
- An equal investment amount is allocated to each selected stock.
- Positions are held for five trading days before being closed.

This strategy provides a baseline for evaluating stock-level predictions without considering broader market conditions.

### Strategy 2: Market-Aware Stock Selection

The second strategy combines stock-level predictions with the Market Regime Model.

The portfolio allocation is adjusted according to the predicted market condition:

- **Bullish:** Full allocation to the selected stocks.
- **Sideways:** Reduced allocation to limit market exposure.
- **Bearish:** No new positions are opened.

Within each permitted market condition, the three highest-ranked stocks are selected using their predicted outperformance probabilities.

This approach introduces market awareness into the trading decision process rather than relying exclusively on individual stock predictions.

## 3. Trading Assumptions

Both strategies follow consistent trading assumptions to ensure a meaningful comparison:

- Initial capital: $1,500.
- Maximum of 15 concurrent positions.
- Five-trading-day holding period.
- Equal allocation across selected positions, subject to the available capital and strategy-specific allocation rules.
- A 0.2% round-trip transaction fee.
- Capital is reused after positions are closed.

## 4. Vectorized Simulation

A vectorized simulation was implemented to evaluate the strategies efficiently across the historical test dataset.

The simulation identifies entry and exit dates, calculates investment amounts, estimates gross and net profits, and tracks trading activity.

This approach provides an initial overview of strategy performance, including trade count, win rate, transaction costs, capital utilization, and cumulative returns.

## 5. Exact Simulation with Reinvestment

An additional event-driven simulation was developed to represent portfolio execution more realistically.

Unlike the vectorized approach, this simulation manages available cash and active positions over time.

Its main components include:

- Opening positions subject to available capital and position limits.
- Closing positions when their holding periods end.
- Deducting transaction fees from realized profits.
- Reinvesting available capital in subsequent trades.
- Tracking cash, invested capital, market value, and total portfolio equity.

Portfolio equity is calculated using available cash and the current market value of open positions. This provides a more realistic representation of portfolio performance than simply accumulating individual trade returns.

## 6. Risk Management and Performance Evaluation

Several performance and risk metrics were calculated to assess the strategies beyond total profitability.

The evaluation includes:

- **Total Return:** Measures overall portfolio growth.
- **CAGR:** Estimates the annualized growth rate.
- **Sharpe Ratio:** Evaluates returns relative to portfolio volatility.
- **Maximum Drawdown:** Measures the largest decline from a previous equity peak.
- **Annualized Volatility:** Measures fluctuations in portfolio returns.
- **Rolling Returns:** Examines performance across different investment horizons.
- **Win Rate:** Measures the proportion of profitable trades.

A Stop Loss mechanism was also tested to examine whether limiting individual trade losses improves overall portfolio performance.

The results showed that Stop Loss did not consistently improve the strategies. In particular, it reduced profitability in the Market-Aware strategy without providing a meaningful improvement in maximum drawdown. Therefore, the baseline configuration without Stop Loss was retained for the final comparison.

## 7. Strategy Comparison

Both strategies were evaluated over a common testing period to ensure that their performance was compared across the same market conditions.

The Market-Aware strategy performed better than the baseline Top 3 strategy. It achieved positive returns, a higher Sharpe Ratio, and a lower maximum drawdown.

These findings suggest that incorporating predicted market conditions into stock selection and capital allocation can improve the performance of a stock-level prediction strategy.

However, the improvement does not necessarily indicate that the underlying models can consistently predict market movements or stock outperformance.

## 8. Benchmark Comparison

The S&P/TSX Composite Index was selected as the benchmark to evaluate the strategies against the broader Canadian equity market.

The benchmark was simulated using the same initial capital and common evaluation period, assuming a buy-and-hold investment.

The results showed that the benchmark substantially outperformed both trading strategies.

Although the Market-Aware strategy achieved positive returns and improved upon the baseline strategy, it did not capture the broader market's growth during the testing period.

This comparison highlights an important distinction between building a profitable trading strategy and developing one that outperforms a passive market investment.

## 9. Final Findings

The simulation results provide several important findings:

1. Stock-level predictions alone were insufficient to produce consistently attractive portfolio performance.
2. Incorporating market regime predictions improved stock selection and capital allocation.
3. The Market-Aware strategy demonstrated better profitability and risk-adjusted performance than the baseline strategy.
4. Stop Loss did not improve the overall results under the tested configuration.
5. Both strategies underperformed the S&P/TSX Composite benchmark.

Overall, the simulation demonstrates the value of combining stock-level predictions with market-level information. However, the benchmark comparison indicates that further improvements in predictive accuracy, portfolio allocation, and risk management are necessary before the strategy can be considered competitive with passive market investing.